# trial_getter_3 — Clinical Trial Intelligence Pipeline

**Version:** 3.0 | **Date:** March 2026  
**Spec:** trial_getter_3_spec.docx v0.1

---

## Architecture Summary

Full rebuild of the CTIP data acquisition and compound cleaning pipeline.
Incorporates lessons from two myeloma pipeline runs and a TxGemma 27B vs Mistral 7B benchmark evaluation.

### Cleaning Step Order (Risk Hierarchy)

| Step | Method | Risk Level | Expected Resolution |
|------|--------|------------|--------------------|
| 1 | Deterministic pre-filter (extended rules) | Lowest | ~50 compounds |
| 2 | RxNorm lookup | Lowest | ~200 compounds |
| 3 | ChEMBL lookup (token length guard) | Lowest | ~80 compounds |
| 4 | Fuzzy INN matching (WHO INN list, edit distance ≤ 2) | Low | ~10 compounds |
| 5 | CAR-T classification module | Low | ~30 compounds |
| 6 | NER classification (scispaCy bc5cdr) | Low-Medium | ~50 excluded |
| 7 | Trial context enrichment (title + summary) | N/A | Context only |
| 8 | PubMed context (NER-confirmed DRUG only) | N/A | Context only |
| 9 | TxGemma 27B cleaning (with full context) | Highest | ~80 compounds |
| 10 | Post-cleaning canonical deduplication | N/A | Merge duplicates |
| 11 | Human review GUI (NULL + low-confidence) | Human | Target: <30 |

### Key Changes vs trial_getter_2

- **LLM:** Mistral 7B → TxGemma 27B via LM Studio REST API (not Ollama)
- **Multi-field extraction:** `intervention` + `brief_title` + `brief_summary` mined in parallel
- **New steps:** Fuzzy INN matching (Step 4), CAR-T module (Step 5), NER (Step 6)
- **Extended pre-filter:** sentence fragments, protocol parameters, procedural terms
- **Extended splitter:** `and/plus/with/or` conjunctions, context-aware hyphen splitting
- **Schema additions:** `source`, `ner_class`, `ner_confidence`, `intervention_type`, `prompt_version`
- **Full timing instrumentation** throughout pipeline
- **Post-cleaning canonical deduplication** (Step 10)

---

**Setup:** Create `.env` file with `AACT_USER`, `AACT_PASSWORD`, `NCBI_API_KEY`, `NCBI_EMAIL`

In [1]:
# =============================================================================
# Cell 1 — Imports & Configuration
# =============================================================================

import os
import re
import json
import time
import sqlite3
import unicodedata
from datetime import datetime
from collections import defaultdict
from typing import Optional
import calendar

import pandas as pd
import psycopg2
import requests
import ipywidgets as widgets
from IPython.display import display, HTML
from dotenv import load_dotenv
from Bio import Entrez
from rapidfuzz import fuzz, process as rfprocess

# Optional NER — imported later in Cell 16 to avoid blocking startup
# import spacy

from domain_router import create_routing_table, load_routing_table, resolve_domain

load_dotenv()

# ---------------------------------------------------------------------------
# AACT (ClinicalTrials.gov)
# ---------------------------------------------------------------------------
AACT_CONFIG = {
    "host"     : "aact-db.ctti-clinicaltrials.org",
    "port"     : 5432,
    "database" : "aact",
    "user"     : os.getenv("AACT_USER"),
    "password" : os.getenv("AACT_PASSWORD"),
}

# ---------------------------------------------------------------------------
# NCBI / PubMed
# ---------------------------------------------------------------------------
NCBI_CONFIG = {
    "api_key" : os.getenv("NCBI_API_KEY"),
    "email"   : os.getenv("NCBI_EMAIL"),
}

# ---------------------------------------------------------------------------
# LLM — TxGemma 27B via LM Studio REST API (NOT Ollama)
# ---------------------------------------------------------------------------
LLM_ENDPOINT = "http://127.0.0.1:1234/v1/chat/completions"
LLM_MODEL    = "txgemma-27b-chat"   # exact model name as shown in LM Studio

# ---------------------------------------------------------------------------
# Database paths
# ---------------------------------------------------------------------------
SQLITE_DB  = "trials_v3.db"
PUBMED_DB  = "pubmed_corpus.db"

# ---------------------------------------------------------------------------
# Pipeline config
# ---------------------------------------------------------------------------
TRIAL_FETCH_LIMIT = 40       # Set to None for no limit
PUBMED_LOOKBACK_YEARS = 1      # change to 20 for full lookback comparison
LLM_BATCH_SIZE        = 15     # TxGemma 27B is 3-4x slower — smaller batches
PROMPT_VERSION        = "v3.0" # bumped whenever system prompt rules change

# ---------------------------------------------------------------------------
# Domain routing
# ---------------------------------------------------------------------------
create_routing_table()
routing_table = load_routing_table()

print("✅ Imports and config loaded")
print(f"   LLM endpoint  : {LLM_ENDPOINT}")
print(f"   LLM model     : {LLM_MODEL}")
print(f"   SQLite DB     : {SQLITE_DB}")
print(f"   PubMed lookback: {PUBMED_LOOKBACK_YEARS} years")
print(f"   Prompt version : {PROMPT_VERSION}")

✅ Routing table written: C:\Users\robin\Programs\Clinical_Trial_2\domain_routing_table.json  (126 entries)
✅ Routing table loaded: 126 entries from C:\Users\robin\Programs\Clinical_Trial_2\domain_routing_table.json
✅ Imports and config loaded
   LLM endpoint  : http://127.0.0.1:1234/v1/chat/completions
   LLM model     : txgemma-27b-chat
   SQLite DB     : trials_v3.db
   PubMed lookback: 1 years
   Prompt version : v3.0


In [2]:
# =============================================================================
# Cell 2 — init_sqlite() — Updated Schema
# =============================================================================
# NEW in v3:
#   compounds      : + source, ner_class, ner_confidence, intervention_type
#   cleaning_log   : + prompt_version
#   trial_compounds: carried forward from v2 — unchanged
# =============================================================================

def init_sqlite():
    """
    (Re)create all SQLite tables.  DESTRUCTIVE — call only when resetting.
    Drops and recreates: trials, corpus, compounds, trial_compounds, cleaning_log,
    compound_abstracts.
    """
    con = sqlite3.connect(SQLITE_DB)
    cur = con.cursor()

    cur.executescript("""
        DROP TABLE IF EXISTS trials;
        DROP TABLE IF EXISTS corpus;
        DROP TABLE IF EXISTS compounds;
        DROP TABLE IF EXISTS trial_compounds;
        DROP TABLE IF EXISTS cleaning_log;
        DROP TABLE IF EXISTS compound_abstracts;

        -- ----------------------------------------------------------------
        -- trials: one row per NCT ID, carries raw AACT fields
        -- ----------------------------------------------------------------
        CREATE TABLE trials (
            nct_id           TEXT PRIMARY KEY,
            brief_title      TEXT,
            brief_summary    TEXT,
            overall_status   TEXT,
            phase            TEXT,
            start_date       TEXT,
            completion_date  TEXT,
            conditions       TEXT,
            interventions    TEXT,
            intervention_types TEXT,   -- v3: preserve DRUG vs BIOLOGICAL
            sponsor          TEXT,
            search_term      TEXT,
            retrieved_at     TEXT
        );

        -- ----------------------------------------------------------------
        -- corpus: formatted text block per trial for context injection
        -- ----------------------------------------------------------------
        CREATE TABLE corpus (
            nct_id       TEXT PRIMARY KEY,
            corpus_text  TEXT,
            created_at   TEXT
        );

        -- ----------------------------------------------------------------
        -- compounds: one row per unique compound candidate
        -- NEW v3 columns: source, ner_class, ner_confidence, intervention_type
        -- ----------------------------------------------------------------
        CREATE TABLE compounds (
            compound_name      TEXT PRIMARY KEY,
            canonical_name     TEXT,
            search_term        TEXT,
            source             TEXT,   -- intervention | title | summary | combined
            intervention_type  TEXT,   -- DRUG | BIOLOGICAL (from AACT)
            ner_class          TEXT,   -- DRUG | PROCEDURE | PROTOCOL | CELL_THERAPY | COMBINATION | UNKNOWN
            ner_confidence     REAL,
            exclude            INTEGER DEFAULT 0,
            exclude_reason     TEXT,
            confidence         TEXT,   -- high | medium | low
            cleaned_by         TEXT,   -- rxnorm | chembl | fuzzy-inn | cart-module | ner-exclude |
                                       -- txgemma | pre-filter | human
            chembl_context     TEXT,
            pubmed_context     TEXT,
            trial_context      TEXT,   -- v3: title+summary context per compound
            combined_context   TEXT,
            retrieved_at       TEXT
        );

        -- ----------------------------------------------------------------
        -- trial_compounds: junction — nct_id ↔ compound_name
        -- ----------------------------------------------------------------
        CREATE TABLE trial_compounds (
            nct_id        TEXT,
            compound_name TEXT,
            source        TEXT,   -- intervention | title | summary
            PRIMARY KEY (nct_id, compound_name, source)
        );

        -- ----------------------------------------------------------------
        -- cleaning_log: one row per LLM batch result
        -- NEW v3: prompt_version column
        -- ----------------------------------------------------------------
        CREATE TABLE cleaning_log (
            id             INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id         TEXT,
            prompt_version TEXT,
            batch_num      INTEGER,
            compound_name  TEXT,
            raw_response   TEXT,
            success        INTEGER,
            error_msg      TEXT,
            elapsed_sec    REAL,
            created_at     TEXT
        );

        -- ----------------------------------------------------------------
        -- compound_abstracts: PubMed title matches per compound
        -- ----------------------------------------------------------------
        CREATE TABLE compound_abstracts (
            compound_name  TEXT,
            pmid           TEXT,
            title          TEXT,
            match_score    REAL,
            PRIMARY KEY (compound_name, pmid)
        );
    """)

    con.commit()
    con.close()
    print(f"✅ SQLite schema ready: {SQLITE_DB}")

print("✅ init_sqlite() defined — call only when resetting database")

✅ init_sqlite() defined — call only when resetting database


In [3]:
# =============================================================================
# Cell 3 — init_sqlite() call
# =============================================================================
# Uncomment to reset. DESTRUCTIVE — drops all existing data.
# init_sqlite()

# Safe non-destructive check
con = sqlite3.connect(SQLITE_DB)
tables = [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()]
con.close()

if tables:
    print(f"✅ Database exists: {SQLITE_DB}")
    print(f"   Tables: {', '.join(tables)}")
else:
    print("⚠️  Database empty — running init_sqlite()")
    init_sqlite()

✅ Database exists: trials_v3.db
   Tables: trials, corpus, compounds, trial_compounds, cleaning_log, sqlite_sequence, compound_abstracts


In [4]:
# =============================================================================
# Cell 4 — fetch_trials() + save_to_sqlite() with timing
# =============================================================================

def fetch_trials(disease_term: str, statuses: list) -> pd.DataFrame:
    """
    Query AACT for trials matching a disease term and list of statuses.
    - Retrieves brief_title, brief_summary, intervention_type alongside names
    - Returns DataFrame
    """
    status_placeholders = ",".join(["%s"] * len(statuses))
    
    # Normalize common search term variants for AACT conditions matching
    _TERM_NORMALIZER = {
        'parkinsons'          : 'parkinson',
        "parkinson's"         : 'parkinson',
        "parkinson's disease" : 'parkinson',
        'alzheimers'          : 'alzheimer',
        "alzheimer's"         : 'alzheimer',
        "alzheimer's disease" : 'alzheimer',
        'als'                 : 'amyotrophic lateral sclerosis',
        'ms'                  : 'multiple sclerosis',
        'dlbcl'               : 'diffuse large b-cell lymphoma',
        'cll'                 : 'chronic lymphocytic leukemia',
        'aml'                 : 'acute myeloid leukemia',
        'nhl'                 : 'non-hodgkin lymphoma',
    }
    disease_term = _TERM_NORMALIZER.get(disease_term.lower().strip(), disease_term)

    query = f"""
        SELECT
            s.nct_id,
            s.brief_title,
            bs.brief_summary,
            s.overall_status,
            s.phase,
            s.start_date,
            s.completion_date,
            c.conditions,
            i.interventions,
            i.intervention_types,
            sp.name AS sponsor
        FROM ctgov.studies s
        LEFT JOIN (
            SELECT nct_id,
                   STRING_AGG(DISTINCT name, ', ') AS conditions
            FROM ctgov.conditions
            WHERE name ILIKE %s
            GROUP BY nct_id
        ) c ON s.nct_id = c.nct_id
        LEFT JOIN (
            SELECT nct_id,
                   STRING_AGG(DISTINCT name, ', ')              AS interventions,
                   STRING_AGG(DISTINCT intervention_type, ', ') AS intervention_types
            FROM ctgov.interventions
            WHERE intervention_type IN ('DRUG', 'BIOLOGICAL')
            GROUP BY nct_id
        ) i ON s.nct_id = i.nct_id
        LEFT JOIN (
            SELECT nct_id, description AS brief_summary
            FROM ctgov.brief_summaries
        ) bs ON s.nct_id = bs.nct_id
        LEFT JOIN ctgov.sponsors sp ON s.nct_id = sp.nct_id
                                   AND sp.lead_or_collaborator = 'lead'
        WHERE c.conditions IS NOT NULL
          AND s.overall_status IN ({status_placeholders})
        ORDER BY s.start_date DESC
        {f"LIMIT {TRIAL_FETCH_LIMIT}" if TRIAL_FETCH_LIMIT else ""}        
    """

    params = [f"%{disease_term}%"] + list(statuses)

    t0 = time.time()
    conn = psycopg2.connect(**AACT_CONFIG)
    df = pd.read_sql(query, conn, params=params)
    conn.close()
    elapsed = time.time() - t0
    print(f"  ⏱  AACT fetch: {elapsed:.1f}s — {len(df)} trials returned")
    return df


def save_to_sqlite(df: pd.DataFrame, search_term: str) -> int:
    """
    Save AACT query results to local SQLite, skipping duplicates.
    Returns total trial count in database.
    """
    df = df.copy()
    df["search_term"]  = search_term.lower().strip()
    df["retrieved_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    con = sqlite3.connect(SQLITE_DB)
    df.to_sql("trials_staging", con, if_exists="replace", index=False)

    con.execute("""
        INSERT OR IGNORE INTO trials (
            nct_id, brief_title, brief_summary, overall_status, phase,
            start_date, completion_date, conditions, interventions,
            intervention_types, sponsor, search_term, retrieved_at
        )
        SELECT
            nct_id, brief_title, brief_summary, overall_status, phase,
            start_date, completion_date, conditions, interventions,
            intervention_types, sponsor, search_term, retrieved_at
        FROM trials_staging
    """)
    con.commit()
    count = con.execute("SELECT COUNT(*) FROM trials").fetchone()[0]
    con.close()
    return count


print("✅ fetch_trials() and save_to_sqlite() defined")

✅ fetch_trials() and save_to_sqlite() defined


In [5]:
# =============================================================================
# Cell 5 — Disease Query GUI
# =============================================================================

disease_input = widgets.Text(
    placeholder='e.g. myeloma, lymphoma, ALS...',
    description='Disease:',
    layout=widgets.Layout(width='450px')
)

status_select = widgets.SelectMultiple(
    options=[
        'RECRUITING',
        'NOT_YET_RECRUITING',
        'ACTIVE_NOT_RECRUITING',
        'COMPLETED',
        'ENROLLING_BY_INVITATION',
        'SUSPENDED',
        'TERMINATED',
        'WITHDRAWN',
    ],
    value=['RECRUITING', 'NOT_YET_RECRUITING', 'ACTIVE_NOT_RECRUITING'],
    description='Status:',
    rows=8,
    layout=widgets.Layout(width='450px')
)

run_button = widgets.Button(
    description='Fetch Trials',
    button_style='primary',
    layout=widgets.Layout(width='150px')
)

output = widgets.Output()

def on_run_click(b):
    with output:
        output.clear_output()
        term     = disease_input.value.strip()
        statuses = list(status_select.value)
        if not term:
            print("⚠️  Please enter a disease term")
            return
        if not statuses:
            print("⚠️  Please select at least one status")
            return

        print(f"🔍 Searching for: '{term}' | Statuses: {statuses}")
        route = resolve_domain(term, routing_table)
        print(f"   Domain: {route.get('domain')} | Subdomain: {route.get('subdomain')} | Conference: {route.get('conference')}")

        df = fetch_trials(term, statuses)
        if df.empty:
            print("⚠️  No trials returned — check disease term and statuses")
            return

        total = save_to_sqlite(df, term)
        print(f"✅ Saved. Total trials in database: {total}")
        display(df.head(5))

run_button.on_click(on_run_click)

# box = widgets.VBox([
#     widgets.HTML('<h3>🔬 CTIP — Trial Getter 3</h3>'),
#     disease_input,
#     status_select,
#     run_button,
#     output
# ])
# display(box)

display(widgets.VBox([
    disease_input,
    status_select,
    run_button,
    output
]))

In [6]:
# =============================================================================
# Cell 6 — Inspect Fetched Trials
# =============================================================================

con = sqlite3.connect(SQLITE_DB)
summary = pd.read_sql("""
    SELECT search_term,
           COUNT(*)              AS trial_count,
           MIN(start_date)       AS earliest,
           MAX(start_date)       AS latest,
           COUNT(interventions)  AS trials_with_interventions
    FROM trials
    GROUP BY search_term
    ORDER BY trial_count DESC
""", con)
con.close()

print("📊 Trials in database by search term:")
display(summary)

📊 Trials in database by search term:


,search_term,trial_count,earliest,latest,trials_with_interventions
0,hiv,40,2024-07-15,2025-09-22,23


In [7]:
# =============================================================================
# Cell 7 — build_corpus()
# =============================================================================

def build_corpus():
    """
    Build formatted corpus text per trial.
    Includes: nct_id, phase, title, interventions, summary.
    Used for context injection into TxGemma prompt.
    """
    con = sqlite3.connect(SQLITE_DB)
    trials = pd.read_sql("SELECT * FROM trials", con)

    records = []
    for _, row in trials.iterrows():
        text = (
            f"NCT_ID: {row['nct_id']}\n"
            f"PHASE: {row['phase']}\n"
            f"TITLE: {row['brief_title']}\n"
            f"INTERVENTIONS: {row['interventions']}\n"
            f"SUMMARY: {str(row['brief_summary'])[:800]}"
        )
        records.append({
            "nct_id"     : row["nct_id"],
            "corpus_text": text,
            "created_at" : datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })

    df_corpus = pd.DataFrame(records)
    df_corpus.to_sql("corpus_staging", con, if_exists="replace", index=False)
    con.execute("""
        INSERT OR IGNORE INTO corpus (nct_id, corpus_text, created_at)
        SELECT nct_id, corpus_text, created_at
        FROM corpus_staging
    """)
    con.commit()
    count = con.execute("SELECT COUNT(*) FROM corpus").fetchone()[0]
    con.close()
    print(f"✅ Corpus built: {count} trials")


build_corpus()

✅ Corpus built: 40 trials


In [10]:
# =============================================================================
# Cell 8 — build_compounds_registry()
# =============================================================================
# MAJOR ARCHITECTURAL CHANGE (v3):
#   - Multi-field extraction: intervention + brief_title + brief_summary
#   - Extended split_interventions() handles: and/plus/with/or, hyphen (contextual)
#   - source column tracks origin of each candidate
#   - intervention_type (DRUG vs BIOLOGICAL) preserved per compound
# =============================================================================

# ---------------------------------------------------------------------------
# Combination splitter — extended for v3
# ---------------------------------------------------------------------------

# Regex for codes that should NOT be split on hyphen (e.g. JNJ-79635322, LCAR-M61D)
_CODE_PATTERN = re.compile(
    r'^[A-Z]{1,6}-[A-Z0-9]{3,}$|^[A-Z]{2,6}\d{3,}$|^[A-Z]+\d+-\d+$',
    re.IGNORECASE
)

# Known INN suffixes (non-exhaustive, used for hyphen-split safety check)
_INN_SUFFIXES = [
    'mab','nib','zib','zumab','ximab','umab','mumab','tinib','ciclib',
    'rafenib','lisib','prazole','vir','vudine','clovir','cillin','mycin',
    'cycline','platin','taxel','mustine','uracil','azole','conazole',
    'vastatin','sartan','pril','olol','lukast','steride','ximab','tomide',
    'pomide','lenide','thalidomide','autoleucel','leucel','celcel',
]

def _looks_like_drug_name(s: str) -> bool:
    """Heuristic: does this token look like a drug name (not a code)?"""
    s = s.strip()
    if len(s) < 4:
        return False
    if _CODE_PATTERN.match(s):
        return False
    sl = s.lower()
    if any(sl.endswith(suf) for suf in _INN_SUFFIXES):
        return True
    # Title-case word ≥ 5 chars — likely a drug name
    if len(s) >= 5 and s[0].isupper() and s[1:].islower():
        return True
    return False


def split_interventions(raw: str) -> list:
    """
    Split a raw intervention string into individual compound candidates.

    Handles:
      - Comma separator          : 'Bortezomib, Lenalidomide'
      - Plus separator           : 'Bortezomib + Lenalidomide'
      - Slash separator          : 'Bortezomib/Lenalidomide'   (with guard)
      - Word conjunctions        : 'Bortezomib and Lenalidomide'
                                   'Curcumin plus Piperine'
                                   'A with B', 'A or B'
      - Hyphen separator         : 'Bortezomib-bendamustine-melphalan'
                                   Only splits if both sides look like drug names,
                                   NOT sponsor codes (JNJ-79635322, LCAR-M61D)
    """
    if not raw or not isinstance(raw, str):
        return []

    s = raw.strip()

    # Step 1: word conjunctions — replace with comma to unify processing
    # Use word boundaries to avoid splitting inside names like 'Venetoclax-and-'
    s = re.sub(r'\b(?:plus|with|and|or)\b', ',', s, flags=re.IGNORECASE)

    # Step 2: plus/slash separators → comma
    s = re.sub(r'\s*[+/]\s*', ',', s)

    # Step 3: split on comma
    parts = [p.strip() for p in s.split(',') if p.strip()]

    # Step 4: contextual hyphen splitting on each part
    final = []
    for part in parts:
        # Only attempt hyphen split if it contains a hyphen and is not a known code
        if '-' in part and not _CODE_PATTERN.match(part):
            tokens = part.split('-')
            # Split only if ALL tokens look like drug names (2+ tokens, each ≥ 4 chars)
            if len(tokens) >= 2 and all(_looks_like_drug_name(t) for t in tokens):
                final.extend(t.strip() for t in tokens if t.strip())
                continue
        final.append(part)

    # Step 5: clean each token — strip trailing injection/solution/dose noise
    _SUFFIX_NOISE = re.compile(
        r'\s+(injection|infusion|solution|tablets?|capsules?|oral|sc|iv|'
        r'intravenous|subcutaneous|for\s+\w+|'
        r'immediate[\s-]release|sustained[\s-]release|extended[\s-]release|'
        r'modified[\s-]release|controlled[\s-]release|delayed[\s-]release|'
        r'\d+\s*hr\b|\d+\s*hour\b|'
        r'24\s*hr|12\s*hr|8\s*hr'
        r')\s*$'
        r'|\s+(?:xr|er|sr|ir|cr|mr)\s*$',
        re.IGNORECASE
    )
    cleaned = []
    for tok in final:
        tok = _SUFFIX_NOISE.sub('', tok).strip()
        # Dose-cohort collapse: strip trailing dose notation
        # e.g. 'Compound 4 mg/m2', 'Compound (N x 10^6 cells/kg)', 'Compound Dose -1'
        tok = re.sub(r'\s+\d+\.?\d*\s*(mg|mcg|g|ml|ng|µg)[\/\w]*.*$', '', tok, flags=re.IGNORECASE)
        tok = re.sub(r'\(\s*[\d\.]+\s*[xX×]\s*10.*?\)', '', tok).strip()
        tok = re.sub(r'\s+[Dd]ose\s*[-–]?\d+.*$', '', tok).strip()
        tok = re.sub(r'\s+\d+\.?\d*\s*$', '', tok).strip()
        if tok and len(tok) >= 2:
            cleaned.append(tok)

    return list(dict.fromkeys(cleaned))  # deduplicate, preserve order


# ---------------------------------------------------------------------------
# NER-based name extraction from free text (title / summary)
# Used here as a lightweight regex fallback before scispaCy loads in Cell 16
# ---------------------------------------------------------------------------

def extract_drug_candidates_from_text(text: str) -> list:
    """
    Lightweight pattern-based extraction of drug name candidates from
    brief_title or brief_summary.  scispaCy NER (Cell 16) provides a
    higher-quality pass; this ensures we don't miss obvious INNs.

    Matches:
      - Title-case words ≥ 5 chars ending in known INN suffixes
      - All-caps codes ≥ 3 chars (sponsor codes)
      - Parenthetical INN aliases: 'brand (INN)'
    """
    if not text:
        return []

    candidates = []

    # INN-suffix words
    for m in re.finditer(r'\b([A-Z][a-z]{3,}(?:' + '|'.join(_INN_SUFFIXES) + r'))\b', text):
        candidates.append(m.group(1))

    # Parenthetical aliases  e.g. 'Revlimid (lenalidomide)'
    for m in re.finditer(r'\(([A-Za-z][a-z]{4,})\)', text):
        candidates.append(m.group(1))

    # Sponsor codes  e.g. JNJ-79635322, ABBV-CLS-484, LCAR-M61D
    for m in re.finditer(r'\b([A-Z]{2,6}-[A-Z0-9]{3,}(?:-[A-Z0-9]+)*)\b', text):
        candidates.append(m.group(1))

    return list(dict.fromkeys(c.strip() for c in candidates if len(c.strip()) >= 4))


# ---------------------------------------------------------------------------
# Main registry builder
# ---------------------------------------------------------------------------

def build_compounds_registry(search_term: str) -> int:
    """
    Extract unique compound candidates from three fields in parallel:
      1. interventions (primary source)
      2. brief_title
      3. brief_summary

    Populates compounds table and trial_compounds junction table.
    Source column records origin of each candidate.

    Returns total compounds in registry.
    """
    term = search_term.lower().strip()
    con  = sqlite3.connect(SQLITE_DB)

    trials = pd.read_sql("""
        SELECT nct_id, brief_title, brief_summary, interventions, intervention_types
        FROM trials
        WHERE search_term = ?
    """, con, params=(term,))

    if trials.empty:
        print(f"⚠️  No trials found for '{term}'. Run GUI search first.")
        con.close()
        return 0

    print(f"📋 Building registry from {len(trials)} trials...")

    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    compound_source: dict  = {}  # compound_name → source label
    compound_itype:  dict  = {}  # compound_name → intervention_type
    # junction: list of (nct_id, compound_name, source)
    junction_rows: list = []

    for _, row in trials.iterrows():
        nct_id         = row["nct_id"]
        interventions  = row["interventions"] if pd.notna(row["interventions"]) else ""
        title   = row["brief_title"]    if pd.notna(row["brief_title"])    else ""
        summary = row["brief_summary"]  if pd.notna(row["brief_summary"])  else ""
        itype   = row["intervention_types"] if pd.notna(row["intervention_types"]) else ""

        # --- Source 1: interventions (primary) ---
        for raw_part in interventions.split(','):
            for cand in split_interventions(raw_part.strip()):
                cand = cand.strip()
                if cand and len(cand) >= 2:
                    if cand not in compound_source:
                        compound_source[cand] = "intervention"
                        compound_itype[cand]  = itype
                    junction_rows.append((nct_id, cand, "intervention"))

        # --- Source 2: brief_title ---
        for cand in extract_drug_candidates_from_text(title):
            cand = cand.strip()
            if cand and len(cand) >= 4:
                if cand not in compound_source:
                    compound_source[cand] = "title"
                    compound_itype[cand]  = ""
                junction_rows.append((nct_id, cand, "title"))

        # --- Source 3: brief_summary ---
        for cand in extract_drug_candidates_from_text(summary):
            cand = cand.strip()
            if cand and len(cand) >= 4:
                if cand not in compound_source:
                    compound_source[cand] = "summary"
                    compound_itype[cand]  = ""
                junction_rows.append((nct_id, cand, "summary"))

    # Write compounds
    new_count = 0
    for cname, source in compound_source.items():
        exists = con.execute(
            "SELECT 1 FROM compounds WHERE compound_name = ?", (cname,)
        ).fetchone()
        if not exists:
            con.execute("""
                INSERT INTO compounds (
                    compound_name, search_term, source, intervention_type, retrieved_at
                ) VALUES (?, ?, ?, ?, ?)
            """, (cname, term, source, compound_itype.get(cname, ""), now))
            new_count += 1

    # Write junction rows (deduplicate)
    seen_junctions = set()
    for nct_id, cname, source in junction_rows:
        key = (nct_id, cname, source)
        if key not in seen_junctions:
            con.execute("""
                INSERT OR IGNORE INTO trial_compounds (nct_id, compound_name, source)
                VALUES (?, ?, ?)
            """, (nct_id, cname, source))
            seen_junctions.add(key)

    con.commit()
    total = con.execute("SELECT COUNT(*) FROM compounds").fetchone()[0]
    jct   = con.execute("SELECT COUNT(*) FROM trial_compounds").fetchone()[0]
    con.close()

    print(f"  New compounds registered : {new_count}")
    print(f"  Total in registry        : {total}")
    print(f"  Junction rows            : {jct}")
    return total


total = build_compounds_registry(disease_input.value)
print(f"✅ Compounds registry ready: {total} compounds")

📋 Building registry from 40 trials...
  New compounds registered : 69
  Total in registry        : 69
  Junction rows            : 106
✅ Compounds registry ready: 69 compounds


In [11]:
# =============================================================================
# Cell 9 — Inspect Registry + Junction Table
# =============================================================================

con = sqlite3.connect(SQLITE_DB)

print("--- Compounds by source ---")
display(pd.read_sql("""
    SELECT source, COUNT(*) AS count
    FROM compounds
    GROUP BY source ORDER BY count DESC
""", con))

print("\n--- First 20 compounds ---")
display(pd.read_sql("""
    SELECT compound_name, source, intervention_type, canonical_name, confidence
    FROM compounds
    ORDER BY compound_name
    LIMIT 20
""", con))

print("\n--- Junction table sample ---")
display(pd.read_sql("""
    SELECT tc.nct_id, tc.compound_name, tc.source, t.brief_title
    FROM trial_compounds tc
    JOIN trials t ON tc.nct_id = t.nct_id
    LIMIT 10
""", con))

con.close()

--- Compounds by source ---


,source,count
0,intervention,57
1,title,7
2,summary,5



--- First 20 compounds ---


,compound_name,source,intervention_type,canonical_name,confidence
0,3TC,intervention,DRUG,None,None
1,Antiretroviral Combinations,intervention,DRUG,None,None
2,Azithromycin,intervention,DRUG,None,None
3,BCG Vaccine Mix for percutaneous use,intervention,BIOLOGICAL,None,None
4,BIC,intervention,DRUG,None,None
5,Baricitinib,title,,None,None
6,Bariticinib,intervention,DRUG,None,None
7,Barticinib,summary,,None,None
8,Bictegravir,intervention,DRUG,None,None
9,Biktarvy,summary,,None,None



--- Junction table sample ---


,nct_id,compound_name,source,brief_title
0,NCT07028385,Bariticinib,intervention,Safety and Impact of Baricitinib on Cell Suriv...
1,NCT07028385,Baricitinib,title,Safety and Impact of Baricitinib on Cell Suriv...
2,NCT07028385,Barticinib,summary,Safety and Impact of Baricitinib on Cell Suriv...
3,NCT07182838,HIV Therapeutic Biologic Mix - HIV GP160,intervention,Proof-of-Concept Clinical Pharmacology Trial f...
4,NCT07182838,BCG Vaccine Mix for percutaneous use,intervention,Proof-of-Concept Clinical Pharmacology Trial f...
5,NCT06919016,DV700P-RNA,intervention,A Study to Evaluate the Safety and Immunogenic...
6,NCT06919016,DV701B1.1-RNA,intervention,A Study to Evaluate the Safety and Immunogenic...
7,NCT07062614,Lamivudine,summary,Exploration of the Variability of Exposure to ...
8,NCT07055828,Intranasal ketamine,intervention,Nebulized vs. IV Ketamine in Preventing Shiver...
9,NCT07055828,Ketamine,intervention,Nebulized vs. IV Ketamine in Preventing Shiver...


In [12]:
# =============================================================================
# Cell 10 — Efficacy Monitor + RUN_ID + Timing Helpers
# =============================================================================

import uuid

# Unique ID for this pipeline run — written to cleaning_log
RUN_ID = f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:6]}"
print(f"🆔 Run ID: {RUN_ID}")

# ---------------------------------------------------------------------------
# Pipeline-wide timer
# ---------------------------------------------------------------------------
PIPELINE_START = time.time()
step_timings: dict = {}   # step_name → elapsed_sec

def step_timer(step_name: str):
    """Context manager — records elapsed time for a named pipeline step."""
    import contextlib

    @contextlib.contextmanager
    def _timer():
        t0 = time.time()
        print(f"⏳ Starting: {step_name}")
        yield
        elapsed = time.time() - t0
        step_timings[step_name] = elapsed
        print(f"✅ {step_name} complete — {elapsed:.1f}s")

    return _timer()


# ---------------------------------------------------------------------------
# Efficacy monitor
# ---------------------------------------------------------------------------

def efficacy_snapshot(label: str = ""):
    """Print current cleaning resolution counts."""
    con = sqlite3.connect(SQLITE_DB)
    rows = con.execute("""
        SELECT
            COUNT(*)                                           AS total,
            SUM(CASE WHEN exclude = 1 THEN 1 ELSE 0 END)      AS excluded,
            SUM(CASE WHEN canonical_name IS NOT NULL
                      AND exclude = 0 THEN 1 ELSE 0 END)      AS resolved,
            SUM(CASE WHEN canonical_name IS NULL THEN 1 ELSE 0 END) AS unresolved,
            SUM(CASE WHEN confidence = 'low'
                      AND exclude = 0 THEN 1 ELSE 0 END)      AS low_confidence
        FROM compounds
    """).fetchone()
    con.close()

    total, excluded, resolved, unresolved, low_conf = rows
    tag = f" [{label}]" if label else ""
    print(f"\n📊 Efficacy snapshot{tag}")
    print(f"   Total compounds  : {total}")
    print(f"   Resolved (kept)  : {resolved}")
    print(f"   Excluded         : {excluded}")
    print(f"   Unresolved       : {unresolved}")
    print(f"   Low confidence   : {low_conf}")


efficacy_snapshot("baseline")
print(f"\n✅ Timer and efficacy monitor ready")

🆔 Run ID: run_20260314_152408_f93208

📊 Efficacy snapshot [baseline]
   Total compounds  : 69
   Resolved (kept)  : 0
   Excluded         : 0
   Unresolved       : 69
   Low confidence   : 0

✅ Timer and efficacy monitor ready


In [13]:
# =============================================================================
# Cell 11 — Step 1: Deterministic Pre-filter (Extended)
# =============================================================================
# NEW in v3:
#   - Sentence fragment detection (word count > 8 + contains verb)
#   - Protocol parameter patterns (g/L, mg/dL, IgG level, trough)
#   - Arm/cohort labels (Prophylaxis A/B/C, Arm A/B, Group 1/2)
#   - Procedural terms (transplant, chemotherapy, apheresis, induction)
#   - Diagnostic assay terms (ELISA, ELISPOT, flow cytometry)
#   - Extended dose-notation collapse
#   - Single common clinical/biological words
#   - Disease status terms (relapsed, refractory, smoldering)
#   - Quality of life / assessment instrument codes (QLQ-*, EORTC-*)
#   - Disease risk classifications (HR-SMM, LR-SMM, MGUS)
# =============================================================================

PRE_FILTER_PATTERNS = [
    # --- Numeric / dosing noise ---
    r"^[\d\s\.\,\%\+\-]+$",
    r"\b\d+\s*(mg|mcg|g|ml|ng|µg|mmol|nmol)\\/?(kg|m2|l|dl)?\b",
    r"dose level\s*[-–]?\d",
    r"^dose level",

    # --- Protocol parameters ---
    r"\btrough\b",
    r"\bigg\b.*\blevel\b",
    r"\bg\/l\b",
    r"\bmg\/dl\b",
    r"\bof each \d+-?day cycle\b",
    r"\bcycle\s+\d+\b",

    # --- Arm / cohort / prophylaxis labels ---
    r"^(arm|group|cohort|stage|stratum|tier)\s+[a-z0-9\/]+$",
    r"^prophylaxis\s+[a-z\/]+$",
    r"^(induction|consolidation|maintenance|intensification)\s+(therapy|treatment|arm|phase)$",

    # --- Procedural / transplant terms ---
    r"^(autologous\s+)?stem\s+cell\s+transplant(ation)?$",
    r"^lymphodepleting\s+chemotherapy$",
    r"^(leukapheresis|apheresis)$",
    r"^(lymphodepletion|mobilization)$",

    # --- Diagnostic / assay terms ---
    r"^elisa$",
    r"^(elispot|flow\s+cytometry|pcr|mri|pet|ct\s+scan)$",
    r"^biospecimen\s+collection$",
    r"^functional\s+tests?$",
    
    # --- Diagnostic imaging / quantification trials (title-level signals) ---
    r"\bquantification\s+tool",
    r"\bpet\s+(marker|tracer|ligand|imaging)\b",
    r"\btau\s+pet\b",
    r"\bamyloid\s+pet\b",
    r"\b(imaging|diagnostic)\s+(study|trial|validation|tool)\b",
    r"\bbiomarker\s+(validation|quantification|study)\b",
    r"\b18f[-\s][a-z0-9]+\b",           # 18F-labelled PET tracers e.g. 18F-PI-2620

    # --- Generic class descriptors (no specific compound identity) ---
    r"^chemotherapy(\s+drug)?$",
    r"^monoclonal\s+antibody$",
    r"^bispecific\s+antibody$",
    r"^standard\s+of\s+care",
    r"^(supportive|palliative)\s+care",
    r"^cancer\s+therapeutic\s+procedure$",
    r"^combination\s+of\s+treatments?$",
    r"^non.?intervention$",
    r"^no\s+maintenance$",
    r"^imid\s+(drug|treatment|agent)$",

    # --- Combination sequences (multi-sentence regimens) ---
    r"\bfollowed by\b",
    r"\bin combination with\b",
    r"\b(induction|consolidation) (therapy|phase)\b.*and\b",

    # --- Pure regimen shorthands: 2-5 uppercase letters only ---
    r"^[A-Z]{2,5}$",

    # --- Misc noise ---
    r"^(placebo|vehicle|buffer)$",
    r"^saline\b",                        # catches saline, saline inhalants, saline solution etc.
    r"\bcontrol\b",
    r"^(infusion|injection)\s+of\b",
    r"^collection$",

    # --- Single common clinical words ---
    r"^(relapsed|refractory|relapse|relapses|remission|resistant)$",
    r"^(safety|efficacy|tolerability|toxicity|response|survival)$",
    r"^(antigens?|receptors?|proteins?|antibodies|antibody|cells?|genes?)$",
    r"^(treatment|therapy|regimen|protocol|intervention|procedure|medication)$",
    r"^(patients?|subjects?|participants?|volunteers?|individuals?)$",
    r"^(disease|condition|disorder|syndrome|cancer|tumor|tumour|malignancy)$",
    r"^(myeloma|lymphoma|leukaemia|leukemia|carcinoma|sarcoma)$",
    r"^(multiple\s+myeloma|smoldering\s+myeloma|plasma\s+cell)$",

    # --- Disease status / risk classifications ---
    r"^(hr|lr|ir)[-\s](smm|mgus|mm)$",
    r"^(high|low|intermediate)[-\s]risk\s+(smm|mgus|myeloma)$",
    r"^(smoldering|indolent|asymptomatic)\s+(myeloma|mm|smm)$",
    r"^mgus$",

    # --- Quality of life / patient reported outcome instruments ---
    r"^qlq[-\s]",
    r"^eortc[-\s]",
    r"^(fact|promis|sf-\d+|eq-\d+|vas|nrs|bpi|pgic)[-\s]",
    r"^(qol|hrqol|pro|prom)[-\s\d]",

    # --- Common biological / immunological generic terms ---
    r"^(cytokines?|chemokines?|interleukins?|interferons?)$",
    r"^(t[-\s]?cells?|b[-\s]?cells?|nk[-\s]?cells?|macrophages?)$",
    r"^(cd\d+)$",
    r"^(mhc|hla|tcr|bcr|car)$",
    r"^autoleucel$",
    r"^(allogeneic|autologous)\s+autoleucel$",
    
    # --- Excipients / delivery agents /buffers ---
    r"^(recombinant\s+human\s+)?hyaluronidase",
    r"^rhuph20$",
    r"^halozyme$",
    r"^polysorbate\s*\d*$",
    r"^(sodium|potassium)\s+chloride$",
    r"^(human\s+)?serum\s+albumin$",
    r"^(citrate|phosphate|acetate|histidine|tris)\s+buffer(ed\s+saline)?$",
    r"^sucrose$",
    r"^trehalose$",
    r"^mannitol$",
    r"^sorbitol$",
    r"^glycine$",
    r"^arginine$",
    r"^(sterile\s+)?(water\s+for\s+injection|normal\s+saline|isotonic\s+saline)$",
    r"^(0\.9%\s+)?sodium\s+chloride\s+(injection|solution|infusion)?$",
    r"^dextrose(\s+\d+%)?(\s+(solution|injection))?$",
    r"^(lactated\s+)?ringer'?s?(\s+solution)?$",
    r"^benzyl\s+alcohol$",
    r"^(methyl|propyl)\s+paraben$",
    r"^edetate\s+(disodium|sodium)?$",
    r"^edta$",
    r"^cremophor$",
    r"^(polyethylene\s+glycol|peg)[-\s]?\d*$",
    r"^hydroxypropyl\s+(methyl)?cellulose$",
    r"^(microcrystalline\s+)?cellulose$",
    r"^(magnesium|calcium|zinc)\s+stearate$",
]

# Common English verbs that signal a sentence fragment, not a compound name
_VERB_PATTERN = re.compile(
    r'\b(is|are|was|were|will|would|should|may|might|can|could|have|has|had|'
    r'being|evaluate|assess|determine|compare|treat|receive|given|administer|'
    r'undergo|enroll|include|exclude|randomize|stratify|follow)\b',
    re.IGNORECASE
)


def pre_filter_compound(name: str) -> tuple:
    """
    Deterministically identify obvious non-compounds.

    Returns:
        (should_exclude: bool, reason: Optional[str])
    """
    raw   = name.strip()
    lower = raw.lower()

    # Minimum length
    if len(lower) < 2:
        return True, "too short"

    # Pure numeric / percentage
    if re.match(r'^[\d\s\.\,\%]+$', lower):
        return True, "numeric or percentage string"

    # Single word common clinical/biological terms
    word_count = len(lower.split())
    if word_count == 1:
        single_word_exclusions = {
            'relapsed', 'refractory', 'relapse', 'relapses', 'remission',
            'resistant', 'safety', 'efficacy', 'tolerability', 'toxicity',
            'response', 'survival', 'antigens', 'antigen', 'receptors',
            'receptor', 'proteins', 'protein', 'antibodies', 'antibody',
            'cells', 'cell', 'genes', 'gene', 'treatment', 'therapy',
            'regimen', 'protocol', 'intervention', 'procedure', 'medication',
            'patients', 'patient', 'subjects', 'subject', 'participants',
            'participant', 'disease', 'condition', 'disorder', 'syndrome',
            'cancer', 'tumor', 'tumour', 'malignancy', 'myeloma', 'lymphoma',
            'leukemia', 'leukaemia', 'carcinoma', 'sarcoma', 'mgus',
            'cytokines', 'cytokine', 'chemokines', 'chemokine', 'macrophages',
            'macrophage', 'placebo', 'vehicle', 'saline', 'collection',
            'infusion', 'injection','sucrose', 'trehalose', 'mannitol', 'sorbitol', 'glycine',
            'arginine', 'gelatin', 'lactose', 'starch', 'talc', 'edta',
            'cremophor', 'excipient', 'diluent', 'solvent', 'carrier', 'saline', 
            'saline inhalants', 'normal saline', 'sterile saline', 'control', 'controls',
        }
        if lower in single_word_exclusions:
            return True, f"single common clinical/biological word: {lower}"

    # Sentence fragment: long string containing a verb
    if word_count > 8 and _VERB_PATTERN.search(raw):
        return True, "sentence fragment (>8 words + verb detected)"

    # Pattern matching
    for pattern in PRE_FILTER_PATTERNS:
        if re.search(pattern, lower, re.IGNORECASE):
            return True, f"pre-filter: pattern '{pattern}'"

    return False, None


def run_prefilter(search_term: str) -> dict:
    """Run Step 1 deterministic pre-filter over all unresolved compounds."""
    term = search_term.lower().strip()
    con  = sqlite3.connect(SQLITE_DB)

    compounds = pd.read_sql("""
        SELECT compound_name FROM compounds
        WHERE search_term = ? AND canonical_name IS NULL
    """, con, params=(term,))

    excluded = 0
    passing  = 0

    for name in compounds["compound_name"]:
        should_exclude, reason = pre_filter_compound(name)
        if should_exclude:
            con.execute("""
                UPDATE compounds
                SET canonical_name = 'EXCLUDED',
                    exclude        = 1,
                    exclude_reason = ?,
                    confidence     = 'high',
                    cleaned_by     = 'pre-filter'
                WHERE compound_name = ? AND search_term = ?
            """, (reason, name, term))
            excluded += 1
        else:
            passing += 1

    con.commit()
    con.close()

    stats = {"pre_filter_excluded": excluded, "pre_filter_passing": passing}
    print(f"   Pre-filter excluded  : {excluded}")
    print(f"   Passing to Step 2    : {passing}")
    return stats


with step_timer("Step 1: Deterministic pre-filter"):
    prefilter_stats = run_prefilter(disease_input.value)
efficacy_snapshot("after pre-filter")

⏳ Starting: Step 1: Deterministic pre-filter
   Pre-filter excluded  : 12
   Passing to Step 2    : 57
✅ Step 1: Deterministic pre-filter complete — 0.0s

📊 Efficacy snapshot [after pre-filter]
   Total compounds  : 69
   Resolved (kept)  : 0
   Excluded         : 12
   Unresolved       : 57
   Low confidence   : 0


In [14]:
# =============================================================================
# Cell 12 — Step 2: RxNorm Lookup with timing
# =============================================================================

RXNORM_BASE = "https://rxnav.nlm.nih.gov/REST"


def rxnorm_lookup(name: str) -> Optional[dict]:
    """
    Look up a compound name in RxNorm.
    Returns dict with rxcui, canonical_name, or None if not found.
    Strips dosing, formulation, concentration, and biosimilar suffix artifacts.
    """
    try:
        url  = f"{RXNORM_BASE}/drugs.json"
        resp = requests.get(url, params={"name": name}, timeout=10)
        resp.raise_for_status()
        data = resp.json()

        concept_groups = data.get("drugGroup", {}).get("conceptGroup", [])
        for group in concept_groups:
            for concept in group.get("conceptProperties", []):
                rxcui    = concept.get("rxcui", "")
                syn_name = concept.get("name", "")
                if not syn_name:
                    continue

                # Strip malformed prefix artifacts e.g. "{49 (" or "{12 ("
                syn_name = re.sub(r'^\{[\d\s]*\(?\s*', '', syn_name).strip()

                # Strip leading numeric dose prefix e.g. "1.1 ml ", "70 ml "
                syn_name = re.sub(
                    r'^\d+\.?\d*\s*(ml|mg|mcg|ug|units?|iu)\s+',
                    '', syn_name, flags=re.IGNORECASE
                ).strip()

                # Strip trailing dose/formulation/concentration
                syn_name = re.sub(
                    r'\s+\d+\.?\d*\s*(ml|mg|mcg|ug|%|units?|iu|cells?/ml).*$',
                    '', syn_name, flags=re.IGNORECASE
                ).strip()

                # Strip biosimilar 4-letter suffixes e.g. -bcmm, -tgvs, -cqyv
                syn_name = re.sub(
                    r'-[a-z]{4}$', '', syn_name, flags=re.IGNORECASE
                ).strip()

                # Strip salt/formulation suffixes
                syn_name = re.sub(
                    r'\s+(oral|injection|subcutaneous|intravenous|'
                    r'hcl|mesylate|phosphate|sulfate|hydrochloride|'
                    r'acetate|citrate|sodium|potassium|calcium).*$',
                    '', syn_name, flags=re.IGNORECASE
                ).strip()

                # Skip if result is empty or just punctuation
                if not syn_name or not re.search(r'[a-zA-Z]', syn_name):
                    continue

                return {"rxcui": rxcui, "canonical_name": syn_name.lower()}

        return None

    except Exception:
        return None


def run_rxnorm_step(search_term: str) -> dict:
    """Run Step 2: RxNorm lookup over unresolved compounds."""
    term = search_term.lower().strip()
    con  = sqlite3.connect(SQLITE_DB)

    compounds = pd.read_sql("""
        SELECT compound_name FROM compounds
        WHERE search_term = ? AND canonical_name IS NULL
    """, con, params=(term,))

    resolved = 0
    total    = len(compounds)

    for i, name in enumerate(compounds["compound_name"]):
        if i % 50 == 0:
            print(f"   RxNorm: {i}/{total}...", end="\r")

        result = rxnorm_lookup(name)
        if result:
            con.execute("""
                UPDATE compounds
                SET canonical_name = ?,
                    confidence     = 'high',
                    cleaned_by     = 'rxnorm',
                    exclude        = 0
                WHERE compound_name = ? AND search_term = ?
            """, (result["canonical_name"], name, term))
            resolved += 1

        time.sleep(0.05)  # respect RxNorm rate limit

    con.commit()
    con.close()

    print(f"   RxNorm resolved      : {resolved}/{total}")
    return {"rxnorm_resolved": resolved}


with step_timer("Step 2: RxNorm lookup"):
    rxnorm_stats = run_rxnorm_step(disease_input.value)
efficacy_snapshot("after RxNorm")

⏳ Starting: Step 2: RxNorm lookup
   RxNorm resolved      : 20/57
✅ Step 2: RxNorm lookup complete — 26.4s

📊 Efficacy snapshot [after RxNorm]
   Total compounds  : 69
   Resolved (kept)  : 20
   Excluded         : 12
   Unresolved       : 37
   Low confidence   : 0


In [20]:
# =============================================================================
# Cell 13 — Step 3: ChEMBL Lookup with token length guard + timing
# =============================================================================
# NEW in v3:
#   - Minimum 4-character token guard
#   - Reject single common words
#   - Validate match quality: query must appear in returned pref_name/synonyms
# =============================================================================

CHEMBL_BASE = "https://www.ebi.ac.uk/chembl/api/data"

_COMMON_WORDS = {
    'cell', 'cells', 'drug', 'agent', 'therapy', 'treatment',
    'injection', 'infusion', 'antibody', 'protein', 'vaccine',
    'study', 'trial', 'dose', 'group', 'arm', 'cycle'
}


def _chembl_match_quality(query: str, pref_name: str, synonyms: list) -> bool:
    """
    Validate match quality: query must appear in pref_name or synonym
    at ≥80 token_set_ratio.
    """
    q_lower = query.lower()
    if fuzz.token_set_ratio(q_lower, pref_name.lower()) >= 80:
        return True
    for syn in synonyms:
        if fuzz.token_set_ratio(q_lower, syn.lower()) >= 80:
            return True
    return False


def chembl_lookup(name: str) -> Optional[dict]:
    """
    Look up a compound in ChEMBL by preferred name.
    Returns canonical name or None.
    Token length guard: reject if < 4 chars or common single word.
    """
    stripped = name.strip()

    # Token length guard
    if len(stripped) < 4:
        return None

    # Single common word guard
    if stripped.lower() in _COMMON_WORDS:
        return None

    # Single all-numeric token guard
    if re.match(r'^\d+$', stripped):
        return None

    try:
        url  = f"{CHEMBL_BASE}/molecule.json"
        params = {
            "pref_name__icontains": stripped,
            "limit": 3,
            "format": "json"
        }
        resp = requests.get(url, params=params, timeout=15)
        resp.raise_for_status()
        data = resp.json()

        molecules = data.get("molecules", [])
        if not molecules:
            return None

        for mol in molecules:
            pref_name = mol.get("pref_name") or ""
            synonyms  = [
                s.get("molecule_synonym", "")
                for s in mol.get("molecule_synonyms", [])
            ]
            if pref_name and _chembl_match_quality(stripped, pref_name, synonyms):
                return {"canonical_name": pref_name.lower(),
                        "chembl_context": f"ChEMBL pref_name: {pref_name}"}

        return None

    except Exception:
        return None


def run_chembl_step(search_term: str) -> dict:
    """Run Step 3: ChEMBL lookup over unresolved compounds."""
    term = search_term.lower().strip()
    con  = sqlite3.connect(SQLITE_DB)

    compounds = pd.read_sql("""
        SELECT compound_name FROM compounds
        WHERE search_term = ? AND canonical_name IS NULL
    """, con, params=(term,))

    resolved = 0
    total    = len(compounds)

    for i, name in enumerate(compounds["compound_name"]):
        if i % 25 == 0:
            print(f"   ChEMBL: {i}/{total}...", end="\r")

        result = chembl_lookup(name)
        if result:
            con.execute("""
                UPDATE compounds
                SET canonical_name = ?,
                    chembl_context = ?,
                    confidence     = 'high',
                    cleaned_by     = 'chembl',
                    exclude        = 0
                WHERE compound_name = ? AND search_term = ?
            """, (result["canonical_name"], result.get("chembl_context"),
                  name, term))
            resolved += 1

        time.sleep(0.1)  # ChEMBL rate limit

    con.commit()
    con.close()

    print(f"   ChEMBL resolved      : {resolved}/{total}")
    return {"chembl_resolved": resolved}


with step_timer("Step 3: ChEMBL lookup"):
    chembl_stats = run_chembl_step(disease_input.value)
efficacy_snapshot("after ChEMBL")

⏳ Starting: Step 3: ChEMBL lookup
   ChEMBL resolved      : 2/37
✅ Step 3: ChEMBL lookup complete — 464.4s

📊 Efficacy snapshot [after ChEMBL]
   Total compounds  : 69
   Resolved (kept)  : 22
   Excluded         : 12
   Unresolved       : 35
   Low confidence   : 0


In [21]:
# =============================================================================
# Cell 14 — Step 4: Fuzzy INN Matching (WHO INN list)
# =============================================================================
# NEW in v3:
#   - Edit distance pass vs WHO INN list (Levenshtein ≤ 2 for strings > 8 chars)
#   - Catches misspellings like 'Elrantamab' → 'elranatamab'
#   - INN list loaded from who_inn_reference.csv (place in notebook directory)
#   - CSV columns: inn, disease_area, drug_class, notes
#   - Add rows to the CSV to extend coverage — no code changes needed
# =============================================================================

try:
    from Levenshtein import distance as levenshtein_distance
except ImportError:
    def levenshtein_distance(s1, s2):
        if len(s1) < len(s2):
            return levenshtein_distance(s2, s1)
        if not s2:
            return len(s1)
        prev = list(range(len(s2) + 1))
        for i, c1 in enumerate(s1):
            curr = [i + 1]
            for j, c2 in enumerate(s2):
                curr.append(min(prev[j+1]+1, curr[j]+1,
                                prev[j] + (c1 != c2)))
            prev = curr
        return prev[-1]


# ---------------------------------------------------------------------------
# Load INN list from CSV
# ---------------------------------------------------------------------------
INN_CSV_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")),
                             "who_inn_reference.csv")

def load_inn_list(csv_path: str) -> list:
    """
    Load WHO INN reference list from CSV.
    Falls back to an empty list with a warning if file not found.
    CSV must have an 'inn' column. Additional columns (disease_area,
    drug_class, notes) are ignored by the matcher but useful for maintenance.
    """
    if not os.path.exists(csv_path):
        print(f"⚠️  INN reference CSV not found: {csv_path}")
        print("   Fuzzy INN matching will be skipped.")
        print("   Place who_inn_reference.csv in the notebook directory to enable.")
        return []
    
    df = pd.read_csv(csv_path, dtype=str)
    if 'inn' not in df.columns:
        print(f"⚠️  CSV missing 'inn' column — check file format")
        return []
    
    inns = df['inn'].dropna().str.strip().str.lower().tolist()
    # Skip any rows that are notes/class descriptors (flagged in notes column)
    if 'notes' in df.columns:
        exclude_mask = df['notes'].str.contains('class descriptor', na=False)
        inns = df.loc[~exclude_mask, 'inn'].dropna().str.strip().str.lower().tolist()
    
    print(f"✅ INN reference loaded: {len(inns)} entries from {csv_path}")
    if 'disease_area' in df.columns:
        areas = df['disease_area'].value_counts()
        print(f"   Disease areas: {', '.join(areas.index[:8].tolist())}{'...' if len(areas) > 8 else ''}")
    return inns


WHO_INN_LIST = load_inn_list(INN_CSV_PATH)
_INN_SET     = set(WHO_INN_LIST)


# ---------------------------------------------------------------------------
# Fuzzy matcher
# ---------------------------------------------------------------------------

def fuzzy_inn_match(name: str, max_distance: int = 2) -> Optional[str]:
    """
    Find the closest WHO INN to the input name using Levenshtein distance.
    Only applied if input length > 8 chars.
    Returns corrected INN or None.
    """
    if not WHO_INN_LIST:
        return None

    lower = name.lower().strip()

    # Exact match
    if lower in _INN_SET:
        return lower

    # Only attempt fuzzy if string is long enough to reduce false positives
    if len(lower) <= 8:
        return None

    best_match = None
    best_dist  = max_distance + 1

    for inn in WHO_INN_LIST:
        if abs(len(lower) - len(inn)) > max_distance:
            continue
        d = levenshtein_distance(lower, inn)
        if d <= max_distance and d < best_dist:
            best_dist  = d
            best_match = inn

    return best_match


def run_fuzzy_inn_step(search_term: str) -> dict:
    """Run Step 4: Fuzzy INN matching over unresolved compounds."""
    if not WHO_INN_LIST:
        print("⚠️  Skipping fuzzy INN — no reference list loaded")
        return {"fuzzy_inn_resolved": 0}

    term = search_term.lower().strip()
    con  = sqlite3.connect(SQLITE_DB)

    compounds = pd.read_sql("""
        SELECT compound_name FROM compounds
        WHERE search_term = ? AND canonical_name IS NULL
    """, con, params=(term,))

    resolved = 0

    for name in compounds["compound_name"]:
        match = fuzzy_inn_match(name)
        if match:
            con.execute("""
                UPDATE compounds
                SET canonical_name = ?,
                    confidence     = 'medium',
                    cleaned_by     = 'fuzzy-inn',
                    exclude        = 0
                WHERE compound_name = ? AND search_term = ?
            """, (match, name, term))
            resolved += 1

    con.commit()
    con.close()

    print(f"   Fuzzy INN resolved   : {resolved}")
    return {"fuzzy_inn_resolved": resolved}


with step_timer("Step 4: Fuzzy INN matching"):
    fuzzy_stats = run_fuzzy_inn_step(disease_input.value)
efficacy_snapshot("after fuzzy INN")

✅ INN reference loaded: 404 entries from C:\Users\robin\Programs\Clinical_Trial_2\who_inn_reference.csv
   Disease areas: supportive, chemotherapy, myeloma, oncology_breast, rheumatology, haematology_lymphoma, infectious_hiv, haematology_leukaemia...
⏳ Starting: Step 4: Fuzzy INN matching
   Fuzzy INN resolved   : 1
✅ Step 4: Fuzzy INN matching complete — 0.2s

📊 Efficacy snapshot [after fuzzy INN]
   Total compounds  : 69
   Resolved (kept)  : 23
   Excluded         : 12
   Unresolved       : 34
   Low confidence   : 0


In [22]:
# =============================================================================
# Cell 15 — Step 5: CAR-T & Cell Therapy Classification Module
# =============================================================================
# NEW in v3:
#   1. Bare 'autoleucel' → EXCLUDE as class descriptor (Rule 0)
#   2. Contains approved -autoleucel INN → normalize to INN, confidence: high
#   3. Contains sponsor code → preserve, confidence: medium
#   4. Contains specific target antigen → preserve as 'target car-t', confidence: medium
#   5. No specific identifier → EXCLUDE as class descriptor
#   6. NK cell therapies with specific modifier → preserve, confidence: medium
#   7. Bare 'NK cells' / generic NK → EXCLUDE as class descriptor
# =============================================================================

# Approved -autoleucel INNs (updated as new approvals occur)
_AUTOLEUCEL_INNS = {
    "idecabtagene vicleucel"      : "idecabtagene-vicleucel",
    "ciltacabtagene autoleucel"   : "ciltacabtagene-autoleucel",
    "equecabtagene autoleucel"    : "equecabtagene-autoleucel",
    "zevorcabtagene autoleucel"   : "zevorcabtagene-autoleucel",
    "axicabtagene ciloleucel"     : "axicabtagene-ciloleucel",
    "tisagenlecleucel"            : "tisagenlecleucel",
    "lisocabtagene maraleucel"    : "lisocabtagene-maraleucel",
    "brexucabtagene autoleucel"   : "brexucabtagene-autoleucel",
}

# Specific CAR-T target antigens that preserve clinical identity
_CART_TARGET_ANTIGENS = [
    'bcma', 'gprc5d', 'cd19', 'cd38', 'slamf7', 'cs1', 'cd70',
    'fcrl5', 'fcrh5', 'cd3', 'cd138', 'nkbb4', 'cd79b', 'cd20',
]

# Sponsor code prefixes for investigational CAR-T products
_CART_SPONSOR_PREFIXES = [
    r'lcar-', r'bb-', r'jnj-', r'iaso', r'ql-', r'gr-', r'ct-?0',
    r'ft\d', r'kq-', r'cbg', r'hbi', r'sl4', r'ufe-', r'gcart',
    r'gc012', r'rd140', r'car2', r'yk', r'uts',
]

# Generic CAR-T patterns (class descriptor → exclude)
_GENERIC_CART_PATTERNS = [
    r'^car-?t\s*(cells?)?\s*(infusion|therapy|treatment|suspension)?$',
    r'^chimeric\s+antigen\s+receptor\s+(t\s+cell|modified).*$',
    r'^car\s*t[-\s]?cell(s)?$',
    r'^allogeneic\s+car-?t$',
    r'^autologous\s+car-?t$',
    r'^autoleucel$',
    r'^(allogeneic|autologous)\s+autoleucel$',
]

# NK cell therapy — specific modifiers that indicate a named product
_NK_SPECIFIC_MODIFIERS = re.compile(
    r'\b(tgf.?b(eta)?[-\s]?imprint|expanded|engineered|modified|'
    r'donor|allogeneic|autologous|cord.?blood|ipsc|'
    r'fc.?optimized|cd16|activated|primed|memory)\b',
    re.IGNORECASE
)

# Generic NK patterns (class descriptor → exclude)
_GENERIC_NK_PATTERNS = re.compile(
    r'^(allogeneic\s+)?nk[\s-]?cells?$|'
    r'^natural\s+killer\s+cells?$|'
    r'^nk\s+cell\s+(therapy|infusion|treatment)$',
    re.IGNORECASE
)

# CAR-T signal terms
_CART_SIGNAL = re.compile(
    r'\b(car[-\s]?t|chimeric\s+antigen|autoleucel|car19|cart\b)',
    re.IGNORECASE
)

# NK cell signal terms
_NK_SIGNAL = re.compile(
    r'\bnk[\s-]?cells?\b|\bnatural\s+killer\b',
    re.IGNORECASE
)


def classify_cart(name: str) -> Optional[dict]:
    """
    CAR-T and cell therapy decision tree classifier.

    Returns dict:
        {canonical_name, exclude, exclude_reason, confidence, cleaned_by}
    or None if name is not a CAR-T / cell therapy construct.
    """
    lower = name.lower().strip()

    is_cart = _CART_SIGNAL.search(lower) or 'autoleucel' in lower
    is_nk   = _NK_SIGNAL.search(lower)

    if not is_cart and not is_nk:
        return None  # not a cell therapy string

    # =========================================================================
    # NK CELL THERAPY RULES
    # =========================================================================
    if is_nk and not is_cart:

        # Generic NK → exclude
        if _GENERIC_NK_PATTERNS.match(lower):
            return {
                "canonical_name": "EXCLUDED",
                "exclude"       : 1,
                "exclude_reason": "generic NK cell class descriptor",
                "confidence"    : "high",
                "cleaned_by"    : "cart-module",
            }

        # Specific NK product — has a meaningful modifier
        if _NK_SPECIFIC_MODIFIERS.search(lower):
            canonical = re.sub(
                r'\s+(injection|infusion|suspension|cells?\s+infusion).*$',
                '', lower
            ).strip()
            return {
                "canonical_name": canonical,
                "exclude"       : 0,
                "exclude_reason": None,
                "confidence"    : "medium",
                "cleaned_by"    : "cart-module",
            }

        # NK signal present but no rule matched → pass to NER/TxGemma
        return None

    # =========================================================================
    # CAR-T RULES
    # =========================================================================

    # --- Rule 0: Bare autoleucel — class descriptor → exclude ---
    if re.match(r'^(allogeneic\s+|autologous\s+)?autoleucel$', lower):
        return {
            "canonical_name": "EXCLUDED",
            "exclude"       : 1,
            "exclude_reason": "bare autoleucel — class descriptor without specific INN prefix",
            "confidence"    : "high",
            "cleaned_by"    : "cart-module",
        }

    # --- Rule 1: Approved -autoleucel INN ---
    for inn_key, canonical in _AUTOLEUCEL_INNS.items():
        if inn_key in lower or inn_key.replace('-', ' ') in lower:
            return {
                "canonical_name": canonical,
                "exclude"       : 0,
                "exclude_reason": None,
                "confidence"    : "high",
                "cleaned_by"    : "cart-module",
            }

    # Check for -autoleucel suffix not in our list
    if re.search(r'\w+autoleucel', lower):
        canonical = re.sub(r'\s+(injection|cells?|infusion).*$', '', lower).strip()
        return {
            "canonical_name": canonical,
            "exclude"       : 0,
            "exclude_reason": None,
            "confidence"    : "medium",
            "cleaned_by"    : "cart-module",
        }

    # --- Rule 2: Sponsor code ---
    for prefix_pat in _CART_SPONSOR_PREFIXES:
        if re.search(prefix_pat, lower, re.IGNORECASE):
            code_match = re.match(r'([A-Z0-9][A-Z0-9\-]{2,15})', name.strip(), re.IGNORECASE)
            canonical  = code_match.group(1).upper() if code_match else name.strip()
            return {
                "canonical_name": canonical.lower() + " car-t",
                "exclude"       : 0,
                "exclude_reason": None,
                "confidence"    : "medium",
                "cleaned_by"    : "cart-module",
            }

    # --- Rule 3: Specific target antigen ---
    found_targets = []
    for antigen in _CART_TARGET_ANTIGENS:
        if re.search(r'\b' + re.escape(antigen) + r'\b', lower, re.IGNORECASE):
            found_targets.append(antigen.upper())

    if found_targets:
        target_str = "-".join(sorted(set(found_targets)))
        return {
            "canonical_name": f"{target_str.lower()} car-t",
            "exclude"       : 0,
            "exclude_reason": None,
            "confidence"    : "medium",
            "cleaned_by"    : "cart-module",
        }

    # --- Rule 4: Generic CAR-T → exclude ---
    for pat in _GENERIC_CART_PATTERNS:
        if re.match(pat, lower, re.IGNORECASE):
            return {
                "canonical_name": "EXCLUDED",
                "exclude"       : 1,
                "exclude_reason": "generic CAR-T class descriptor without specific target or code",
                "confidence"    : "high",
                "cleaned_by"    : "cart-module",
            }

    # CAR-T signal present but no rule matched → pass to NER/TxGemma
    return None


def run_cart_step(search_term: str) -> dict:
    """Run Step 5: CAR-T & cell therapy classification over unresolved compounds."""
    term = search_term.lower().strip()
    con  = sqlite3.connect(SQLITE_DB)

    compounds = pd.read_sql("""
        SELECT compound_name FROM compounds
        WHERE search_term = ? AND canonical_name IS NULL
    """, con, params=(term,))

    classified = 0
    excluded   = 0

    for name in compounds["compound_name"]:
        result = classify_cart(name)
        if result:
            con.execute("""
                UPDATE compounds
                SET canonical_name = ?,
                    exclude        = ?,
                    exclude_reason = ?,
                    confidence     = ?,
                    cleaned_by     = ?,
                    ner_class      = 'CELL_THERAPY'
                WHERE compound_name = ? AND search_term = ?
            """, (
                result["canonical_name"], result["exclude"],
                result["exclude_reason"], result["confidence"],
                result["cleaned_by"], name, term
            ))
            if result["exclude"]:
                excluded += 1
            else:
                classified += 1

    con.commit()
    con.close()

    print(f"   CAR-T classified     : {classified}")
    print(f"   CAR-T excluded       : {excluded}")
    return {"cart_classified": classified, "cart_excluded": excluded}


with step_timer("Step 5: CAR-T classification"):
    cart_stats = run_cart_step(disease_input.value)
efficacy_snapshot("after CAR-T module")

⏳ Starting: Step 5: CAR-T classification
   CAR-T classified     : 0
   CAR-T excluded       : 0
✅ Step 5: CAR-T classification complete — 0.0s

📊 Efficacy snapshot [after CAR-T module]
   Total compounds  : 69
   Resolved (kept)  : 23
   Excluded         : 12
   Unresolved       : 34
   Low confidence   : 0


In [23]:
# =============================================================================
# Cell 16 — Step 6: NER Classification (scispaCy)
# =============================================================================
# Uses en_ner_bc5cdr_md (biomedical NER trained on BC5CDR).
# Classifies unresolved compounds into:
#   DRUG | PROCEDURE | PROTOCOL | CELL_THERAPY | COMBINATION | UNKNOWN
# PROCEDURE and PROTOCOL are auto-excluded.
# Writes ner_class + ner_confidence to compounds table.
# =============================================================================

# Install:
#   pip install scispacy
#   pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.3/en_ner_bc5cdr_md-0.5.3.tar.gz

try:
    import spacy
    _nlp = spacy.load("en_ner_bc5cdr_md")
    NER_AVAILABLE = True
    print("✅ scispaCy en_ner_bc5cdr_md loaded")
except Exception as e:
    NER_AVAILABLE = False
    print(f"⚠️  scispaCy not available ({e}) — Step 6 will assign UNKNOWN to all")


# NER entity type → pipeline class mapping
_NER_CLASS_MAP = {
    "CHEMICAL": "DRUG",
    "DISEASE" : "PROTOCOL",  # disease mentions are not compounds
}

# Keywords that force-reclassify regardless of NER output
_PROCEDURE_KEYWORDS = {
    'transplant', 'transplantation', 'apheresis', 'leukapheresis',
    'irradiation', 'radiation', 'surgery', 'biopsy', 'aspiration',
    'mobilization', 'conditioning', 'stem cell'
}
_PROTOCOL_KEYWORDS = {
    'induction', 'consolidation', 'maintenance', 'prophylaxis',
    'standard of care', 'regimen', 'protocol'
}


def ner_classify(name: str) -> tuple:
    """
    Classify a compound string using scispaCy NER.

    Returns:
        (ner_class: str, ner_confidence: float)
    """
    lower = name.lower().strip()

    # Keyword overrides (fastest path)
    for kw in _PROCEDURE_KEYWORDS:
        if kw in lower:
            return "PROCEDURE", 0.95
    for kw in _PROTOCOL_KEYWORDS:
        if kw in lower:
            return "PROTOCOL", 0.9

    if not NER_AVAILABLE:
        return "UNKNOWN", 0.0

    doc = _nlp(name)
    ents = doc.ents

    if not ents:
        return "UNKNOWN", 0.5

    # Take the dominant entity type
    type_counts = defaultdict(int)
    for ent in ents:
        mapped = _NER_CLASS_MAP.get(ent.label_, "UNKNOWN")
        type_counts[mapped] += 1

    dominant = max(type_counts, key=type_counts.get)
    total    = sum(type_counts.values())
    conf     = type_counts[dominant] / total

    return dominant, round(conf, 3)


def run_ner_step(search_term: str) -> dict:
    """
    Run Step 6: NER classification.
    - Writes ner_class + ner_confidence to all unresolved compounds
    - Auto-excludes PROCEDURE and PROTOCOL classes
    """
    term = search_term.lower().strip()
    con  = sqlite3.connect(SQLITE_DB)

    compounds = pd.read_sql("""
        SELECT compound_name FROM compounds
        WHERE search_term = ? AND canonical_name IS NULL
    """, con, params=(term,))

    ner_excluded = 0
    ner_labelled = 0

    for name in compounds["compound_name"]:
        ner_class, ner_conf = ner_classify(name)

        if ner_class in ("PROCEDURE", "PROTOCOL"):
            con.execute("""
                UPDATE compounds
                SET canonical_name = 'EXCLUDED',
                    exclude        = 1,
                    exclude_reason = ?,
                    confidence     = 'high',
                    cleaned_by     = 'ner-exclude',
                    ner_class      = ?,
                    ner_confidence = ?
                WHERE compound_name = ? AND search_term = ?
            """, (f"NER class: {ner_class}", ner_class, ner_conf, name, term))
            ner_excluded += 1
        else:
            con.execute("""
                UPDATE compounds
                SET ner_class      = ?,
                    ner_confidence = ?
                WHERE compound_name = ? AND search_term = ?
            """, (ner_class, ner_conf, name, term))
            ner_labelled += 1

    con.commit()
    con.close()

    print(f"   NER labelled         : {ner_labelled}")
    print(f"   NER excluded         : {ner_excluded}")
    return {"ner_labelled": ner_labelled, "ner_excluded": ner_excluded}


with step_timer("Step 6: NER classification"):
    ner_stats = run_ner_step(disease_input.value)
efficacy_snapshot("after NER")

⚠️  scispaCy not available (No module named 'spacy') — Step 6 will assign UNKNOWN to all
⏳ Starting: Step 6: NER classification
   NER labelled         : 34
   NER excluded         : 0
✅ Step 6: NER classification complete — 0.0s

📊 Efficacy snapshot [after NER]
   Total compounds  : 69
   Resolved (kept)  : 23
   Excluded         : 12
   Unresolved       : 34
   Low confidence   : 0


In [24]:
# =============================================================================
# Cell 17 — Step 7: Trial Context Enrichment (title + summary per compound)
# =============================================================================
# NEW in v3: enriches each unresolved compound with the brief_title and
# brief_summary of every trial that uses it.  Written to trial_context column.
# Passed to TxGemma in Step 9 alongside PubMed context.
# =============================================================================

def build_trial_context(search_term: str) -> int:
    """
    For each unresolved compound, aggregate trial titles and
    brief summaries from all associated trials.
    Writes a summary string to compounds.trial_context.

    Returns: number of compounds enriched.
    """
    term = search_term.lower().strip()
    con  = sqlite3.connect(SQLITE_DB)

    # Load unresolved compounds that need context
    compounds = pd.read_sql("""
        SELECT compound_name FROM compounds
        WHERE search_term = ? AND canonical_name IS NULL
    """, con, params=(term,))

    enriched = 0

    for name in compounds["compound_name"]:
        # Find all trials using this compound via junction table
        trial_rows = pd.read_sql("""
            SELECT t.nct_id, t.brief_title, t.brief_summary
            FROM trial_compounds tc
            JOIN trials t ON tc.nct_id = t.nct_id
            WHERE tc.compound_name = ?
            LIMIT 3
        """, con, params=(name,))

        if trial_rows.empty:
            continue

        context_parts = []
        for _, tr in trial_rows.iterrows():
            title   = (tr["brief_title"]   or "").strip()
            summary = (tr["brief_summary"] or "").strip()[:300]
            context_parts.append(
                f"[{tr['nct_id']}] TITLE: {title} | SUMMARY: {summary}"
            )

        context_str = " || ".join(context_parts)

        con.execute("""
            UPDATE compounds
            SET trial_context = ?
            WHERE compound_name = ? AND search_term = ?
        """, (context_str, name, term))
        enriched += 1

    con.commit()
    con.close()

    print(f"   Compounds enriched with trial context: {enriched}")
    return enriched


with step_timer("Step 7: Trial context enrichment"):
    ctx_count = build_trial_context(disease_input.value)
print(f"✅ Trial context ready for {ctx_count} compounds")

⏳ Starting: Step 7: Trial context enrichment
   Compounds enriched with trial context: 34
✅ Step 7: Trial context enrichment complete — 0.0s
✅ Trial context ready for 34 compounds


In [25]:
# =============================================================================
# Cell 17b — collect_pubmed_corpus()
# =============================================================================
# Fetches PubMed abstracts for the current search term over the past
# PUBMED_LOOKBACK_YEARS years.  Stores results in pubmed_corpus.db.
# Safe to re-run — skips PMIDs already in the database.
# Must run before Cell 18 (PubMed context matching).
# =============================================================================

import calendar

def calculate_date_chunks(term: str, start_year: int, end_year: int,
                          target_max: int = 9000) -> list:
    """
    Probe PubMed to determine how many results exist per year (and per
    month for high-volume years), then build chunks that stay safely
    under the 9,999 esearch retmax hard ceiling.
    Returns list of (date_string_start, date_string_end) tuples where
    dates are in YYYY/MM/DD format.
    """
    print("📐 Calculating safe date range chunks...")

    # -- Year-level probe --------------------------------------------------
    year_counts = {}
    for year in range(start_year, end_year + 1):
        date_range = f"{year}/01/01:{year}/12/31[pdat]"
        try:
            handle = Entrez.esearch(
                db     = "pubmed",
                term   = f"{term}[tiab] AND {date_range}",
                retmax = 0
            )
            result = Entrez.read(handle)
            handle.close()
            year_counts[year] = int(result["Count"])
            time.sleep(0.15)
        except Exception as e:
            print(f"  ⚠️  Could not count year {year}: {e}")
            year_counts[year] = 0

    print(f"  {'Year':<8} {'Count':>8}")
    print(f"  {'----':<8} {'-----':>8}")
    for year in sorted(year_counts):
        print(f"  {year:<8} {year_counts[year]:>8}")

    # -- Build chunks, dropping to month-level for high-volume years -------
    chunks = []

    for year in range(start_year, end_year + 1):
        count = year_counts.get(year, 0)

        if count <= target_max:
            # Whole year fits safely in one chunk
            chunks.append((f"{year}/01/01", f"{year}/12/31"))
        else:
            # Year exceeds ceiling — probe monthly and build sub-chunks
            print(f"  ⚠️  {year} has {count} abstracts — probing monthly...")
            month_counts = {}
            for month in range(1, 13):
                month_str  = f"{month:02d}"
                last_day   = calendar.monthrange(year, month)[1]
                date_range = f"{year}/{month_str}/01:{year}/{month_str}/{last_day}[pdat]"
                try:
                    handle = Entrez.esearch(
                        db     = "pubmed",
                        term   = f"{term}[tiab] AND {date_range}",
                        retmax = 0
                    )
                    result = Entrez.read(handle)
                    handle.close()
                    month_counts[month] = int(result["Count"])
                    time.sleep(0.15)
                except Exception as e:
                    print(f"    ⚠️  Could not count {year}/{month_str}: {e}")
                    month_counts[month] = 0

            print(f"  {'Month':<10} {'Count':>8}")
            for m, c in month_counts.items():
                print(f"  {year}/{m:02d}     {c:>8}")

            # Group months into sub-chunks under target_max
            sub_start     = 1
            running_total = 0

            for month in range(1, 13):
                c = month_counts.get(month, 0)
                if running_total + c > target_max and running_total > 0:
                    sub_end      = month - 1
                    last_day_end = calendar.monthrange(year, sub_end)[1]
                    chunks.append((
                        f"{year}/{sub_start:02d}/01",
                        f"{year}/{sub_end:02d}/{last_day_end}"
                    ))
                    sub_start     = month
                    running_total = c
                else:
                    running_total += c

            # Final sub-chunk for the year
            last_day_dec = calendar.monthrange(year, 12)[1]
            chunks.append((
                f"{year}/{sub_start:02d}/01",
                f"{year}/12/{last_day_dec}"
            ))

    print(f"\n  Generated {len(chunks)} chunk(s):")
    for i, (cs, ce) in enumerate(chunks):
        print(f"  Chunk {i+1}: {cs} → {ce}")

    return chunks


def fetch_chunk(term: str, chunk_start: str, chunk_end: str,
                existing: set, con, now: str) -> tuple:
    """
    Fetch all PubMed abstracts for a single date range chunk.
    Phase 1: collect all PMIDs via single esearch (chunks guaranteed
             <9000 by calculate_date_chunks — no paging needed).
    Phase 2: fetch abstracts in batches using explicit PMID lists.
    Returns (new_count, skipped_count).
    """
    date_range = f"{chunk_start}:{chunk_end}[pdat]"

    # ------------------------------------------------------------------
    # Phase 1 — Collect all PMIDs via single esearch call.
    # No usehistory, no uidlist, no history server involved.
    # calculate_date_chunks guarantees every chunk is under 9,000 so
    # retmax=9000 retrieves the full PMID list in one round trip.
    # ------------------------------------------------------------------
    print(f"  🔎 Phase 1: collecting PMIDs...")

    for attempt in range(3):
        try:
            handle = Entrez.esearch(
                db     = "pubmed",
                term   = f"{term}[tiab] AND {date_range}",
                retmax = 9000
            )
            search_results = Entrez.read(handle)
            handle.close()
            time.sleep(0.15)
            break
        except Exception as e:
            print(f"  ⚠️  esearch attempt {attempt+1}/3: {e}")
            if attempt < 2:
                time.sleep(10)
            else:
                print(f"  ❌ esearch permanently failed — skipping chunk")
                return 0, 0

    chunk_total = int(search_results["Count"])
    all_pmids   = search_results["IdList"]

    print(f"  📊 {chunk_total} abstracts in this chunk")
    print(f"  ✅ Phase 1 complete — {len(all_pmids)} PMIDs collected")

    if len(all_pmids) == 0:
        return 0, 0

    # ------------------------------------------------------------------
    # Phase 2 — Fetch abstracts by explicit PMID list.
    # No webenv/query_key — no history server, no retstart ceiling.
    # Dynamic batch sizing: smaller batches for large chunks.
    # ------------------------------------------------------------------
    print(f"  📥 Phase 2: fetching abstracts...")

    if chunk_total > 5000:
        batch_size = 100
        print(f"  ℹ️  Large chunk ({chunk_total} abstracts) — using batch_size=100")
    else:
        batch_size = 200

    new_count     = 0
    skipped_count = 0
    total_batches = (len(all_pmids) + batch_size - 1) // batch_size

    for batch_num, batch_start in enumerate(range(0, len(all_pmids), batch_size), start=1):
        pmid_batch = all_pmids[batch_start : batch_start + batch_size]
        records    = None

        for attempt in range(3):
            try:
                handle = Entrez.efetch(
                    db      = "pubmed",
                    rettype = "xml",
                    retmode = "xml",
                    id      = ",".join(pmid_batch)
                )
                records = Entrez.read(handle)
                handle.close()
                break
            except Exception as e:
                print(f"\n    ⚠️  Batch {batch_num}/{total_batches} attempt {attempt+1}/3: {e}")
                if attempt < 2:
                    time.sleep(10)
                else:
                    print(f"\n    ❌ Batch {batch_num} permanently failed: {e}")
                    records = None

        time.sleep(0.15)

        if records is None:
            continue

        for record in records.get("PubmedArticle", []):
            try:
                article = record["MedlineCitation"]["Article"]
                pmid    = str(record["MedlineCitation"]["PMID"])

                if pmid in existing:
                    continue

                title = str(article.get("ArticleTitle", ""))

                abstract_obj   = article.get("Abstract", {})
                abstract_texts = abstract_obj.get("AbstractText", [])
                if isinstance(abstract_texts, list):
                    abstract = " ".join(str(t) for t in abstract_texts)
                else:
                    abstract = str(abstract_texts)

                mesh_list  = record["MedlineCitation"].get("MeshHeadingList", [])
                mesh_terms = "; ".join(
                    str(m["DescriptorName"]) for m in mesh_list
                ) if mesh_list else None

                author_list = article.get("AuthorList", [])
                authors = "; ".join(
                    f"{a.get('LastName', '')} {a.get('ForeName', '')}".strip()
                    for a in author_list if "LastName" in a
                ) if author_list else None

                journal = str(article.get("Journal", {}).get("Title", ""))

                pub_date_obj = (
                    article.get("Journal", {})
                            .get("JournalIssue", {})
                            .get("PubDate", {})
                )
                pub_date = (
                    pub_date_obj.get("Year") or
                    pub_date_obj.get("MedlineDate") or
                    "Unknown"
                )

                con.execute("""
                    INSERT OR IGNORE INTO abstracts (
                        pmid, search_term, title, abstract,
                        mesh_terms, authors, journal,
                        pub_date, retrieved_at
                    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
                """, (
                    pmid, term, title, abstract,
                    mesh_terms, authors, journal,
                    str(pub_date), now
                ))
                new_count += 1
                existing.add(pmid)

            except Exception:
                skipped_count += 1
                continue

        con.commit()
        print(f"    Batch {batch_num}/{total_batches} — {new_count} new so far", end="\r")

    return new_count, skipped_count


def collect_pubmed_corpus(search_term: str) -> int:
    """
    Fetch all PubMed abstracts for the given disease search term
    over the past PUBMED_LOOKBACK_YEARS years.
    Stores results in pubmed_corpus.db.
    Returns total abstract count.
    """
    term = search_term.lower().strip()

    Entrez.api_key = NCBI_CONFIG["api_key"]
    Entrez.email   = NCBI_CONFIG["email"]

    end_year   = datetime.now().year
    start_year = end_year - PUBMED_LOOKBACK_YEARS

    print(f"🔍 Searching PubMed for: '{term}'")
    print(f"📅 Date range: {start_year}–{end_year} ({PUBMED_LOOKBACK_YEARS} years)")

    chunks = calculate_date_chunks(term, start_year, end_year)

    con = sqlite3.connect(PUBMED_DB)
    con.execute("""
        CREATE TABLE IF NOT EXISTS abstracts (
            pmid         TEXT PRIMARY KEY,
            search_term  TEXT,
            title        TEXT,
            abstract     TEXT,
            mesh_terms   TEXT,
            authors      TEXT,
            journal      TEXT,
            pub_date     TEXT,
            retrieved_at TEXT
        )
    """)
    con.commit()

    existing = set(
        row[0] for row in
        con.execute("SELECT pmid FROM abstracts WHERE search_term = ?", (term,))
    )
    print(f"\n⏭️  Already in database: {len(existing)}")

    now           = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    total_new     = 0
    total_skipped = 0

    for i, (chunk_start, chunk_end) in enumerate(chunks):
        print(f"\n📦 Chunk {i+1}/{len(chunks)}: {chunk_start} → {chunk_end}")
        new, skipped  = fetch_chunk(term, chunk_start, chunk_end, existing, con, now)
        total_new     += new
        total_skipped += skipped
        print(f"  ✅ Chunk {i+1} done — {new} new abstracts")

    total_count = con.execute(
        "SELECT COUNT(*) FROM abstracts WHERE search_term = ?", (term,)
    ).fetchone()[0]
    con.close()

    print(f"\n✅ PubMed collection complete")
    print(f"   New abstracts added  : {total_new}")
    print(f"   Skipped              : {total_skipped}")
    print(f"   Total in corpus      : {total_count}")
    return total_count


with step_timer("Step 7b: PubMed corpus collection"):
    pubmed_total = collect_pubmed_corpus(disease_input.value)
print(f"✅ PubMed corpus ready: {pubmed_total} abstracts for '{disease_input.value}'")

⏳ Starting: Step 7b: PubMed corpus collection
🔍 Searching PubMed for: 'hiv'
📅 Date range: 2025–2026 (1 years)
📐 Calculating safe date range chunks...
  Year        Count
  ----        -----
  2025        15426
  2026         3693
  ⚠️  2025 has 15426 abstracts — probing monthly...
  Month         Count
  2025/01         3550
  2025/02         1416
  2025/03         1533
  2025/04         1559
  2025/05         1468
  2025/06         1493
  2025/07         1653
  2025/08         1438
  2025/09         1590
  2025/10         1617
  2025/11         1637
  2025/12         1912

  Generated 4 chunk(s):
  Chunk 1: 2025/01/01 → 2025/04/30
  Chunk 2: 2025/05/01 → 2025/09/30
  Chunk 3: 2025/10/01 → 2025/12/31
  Chunk 4: 2026/01/01 → 2026/12/31

⏭️  Already in database: 0

📦 Chunk 1/4: 2025/01/01 → 2025/04/30
  🔎 Phase 1: collecting PMIDs...
  📊 7038 abstracts in this chunk
  ✅ Phase 1 complete — 7038 PMIDs collected
  📥 Phase 2: fetching abstracts...
  ℹ️  Large chunk (7038 abstracts) — using b

In [26]:
# =============================================================================
# Cell 18 — Step 8: PubMed Context (NER-confirmed DRUG candidates only)
# =============================================================================
# v3 change: PubMed context only fetched for ner_class = 'DRUG' | 'UNKNOWN'
# This avoids wasting API calls on PROCEDURE/PROTOCOL already excluded by NER.
# Also fixes v2 bug: empty string vs NULL check corrected.
# =============================================================================

def build_pubmed_context_for_compounds(search_term: str) -> int:
    """
    For each unresolved compound with ner_class DRUG or UNKNOWN,
    find matching PubMed abstracts and write titles to compound_abstracts.

    Uses fuzzy match (token_set_ratio ≥ 75) between compound name
    and abstract title.

    Returns: number of compounds matched.
    """
    term = search_term.lower().strip()

    con_trials = sqlite3.connect(SQLITE_DB)
    con_pubmed = sqlite3.connect(PUBMED_DB)

    # Load target compounds — NER DRUG or UNKNOWN only
    compounds = pd.read_sql("""
        SELECT compound_name FROM compounds
        WHERE search_term = ?
          AND canonical_name IS NULL
          AND (ner_class IN ('DRUG', 'UNKNOWN') OR ner_class IS NULL)
    """, con_trials, params=(term,))

    # Load PubMed abstracts (titles only — keep memory footprint small)
    try:
        abstracts = pd.read_sql("""
            SELECT pmid, title FROM abstracts
            WHERE search_term = ?
              AND title IS NOT NULL
              AND TRIM(title) != ''
        """, con_pubmed, params=(term,))
    except Exception:
        abstracts = pd.DataFrame(columns=["pmid", "title"])
        print("⚠️  PubMed corpus not found — skipping PubMed context")

    if abstracts.empty:
        print("⚠️  No PubMed abstracts available for context matching")
        con_trials.close()
        con_pubmed.close()
        return 0

    print(f"   Matching {len(compounds)} compounds vs {len(abstracts)} abstracts...")

    matched = 0
    titles  = abstracts["title"].tolist()
    pmids   = abstracts["pmid"].tolist()

    t0 = time.time()

    for i, name in enumerate(compounds["compound_name"]):
        if i % 20 == 0 and i > 0:
            elapsed = time.time() - t0
            rate    = i / elapsed
            eta     = (len(compounds) - i) / max(rate, 0.01)
            print(f"   Progress: {i}/{len(compounds)} | rate: {rate:.1f}/s | ETA: {eta:.0f}s", end="\r")

        # rapidfuzz — find top-2 matches above threshold
        hits = rfprocess.extract(
            name, titles,
            scorer=fuzz.token_set_ratio,
            limit=2, score_cutoff=75
        )

        for title, score, idx in hits:
            pmid = pmids[idx]
            con_trials.execute("""
                INSERT OR IGNORE INTO compound_abstracts
                    (compound_name, pmid, title, match_score)
                VALUES (?, ?, ?, ?)
            """, (name, pmid, title, score))

        if hits:
            matched += 1

    con_trials.commit()
    con_trials.close()
    con_pubmed.close()

    print(f"\n   Compounds with PubMed context: {matched}")
    return matched


with step_timer("Step 8: PubMed context"):
    pubmed_matched = build_pubmed_context_for_compounds(disease_input.value)
print(f"✅ PubMed context built for {pubmed_matched} compounds")

⏳ Starting: Step 8: PubMed context
   Matching 34 compounds vs 17976 abstracts...
   Progress: 20/34 | rate: 17.7/s | ETA: 1s
   Compounds with PubMed context: 4
✅ Step 8: PubMed context complete — 2.0s
✅ PubMed context built for 4 compounds


In [27]:
# =============================================================================
# Cell 19 — Step 9: TxGemma 27B Cleaning
# =============================================================================
# LM Studio REST API — NOT Ollama.
# Per-batch timer + rolling ETA.
# Full context injection: trial_context + PubMed abstract titles.
# Prompt v3 rules:
#   - Do NOT resolve class descriptors to specific drugs
#   - Do NOT hallucinate context-specific abbreviation expansions
#   - Exclude generic CAR-T strings without target
#   - Strip dose/formulation, preserve sponsor codes
#   - Return combinations as split components joined with ' + '
# =============================================================================

CLEANING_SYSTEM_PROMPT = f"""\
You are a clinical pharmacology expert normalizing intervention strings from
clinical trial databases into canonical compound names.

RULES:
1. Return canonical INN (International Nonproprietary Name) in lowercase.
2. Strip formulation noise: brand names → INN, route/dose suffixes removed.
3. Combination strings: split into components joined with ' + '.
4. Sponsor codes (JNJ-XXXXX, ABBV-XXX, CLN-XXX, AMG-XXX etc.): preserve exactly.
   If uncertain about a sponsor code, return canonical_name=null and confidence=null
   rather than excluding — a human will review it.
5. CAR-T with specific target: return as 'target car-t' (e.g. 'bcma car-t').
6. CAR-T without specific target (e.g. 'CAR-T cells', 'CAR-T treatment'):
   EXCLUDE — set exclude=true, reason='generic CAR-T class descriptor'.
7. Class descriptors (Anti-CD38 MAb, PD-1 Inhibitor, Beta-Adrenergic Antagonist):
   EXCLUDE — do NOT resolve to a specific drug. These are not compound identities.
8. Protocol strings, procedural terms, lab parameters: EXCLUDE.
9. Context-specific abbreviations (VRD, DVRd, KPD): resolve only if trial context
   explicitly spells out the components. If uncertain, EXCLUDE.
   IMPORTANT: Do not guess abbreviation expansions — e.g. 'CarCyDex' in myeloma
   context is likely carfilzomib+cyclophosphamide+dexamethasone, NOT carmustine+
   cytarabine+dexamethasone. Always prefer the disease-context expansion.
10. Confidence: 'high' = certain INN; 'medium' = inferred with good evidence;
    'low' = uncertain — prefer NULL + human review over a guess.
11. If you are not confident, set canonical_name=null and confidence='low'.
    Do not hallucinate a plausible-sounding name.

Prompt version: {PROMPT_VERSION}
"""


def get_context_for_batch(compound_names: list, con) -> str:
    """
    Retrieve trial context and PubMed abstract titles for a batch.
    Returns formatted string for injection into the prompt.
    """
    placeholders = ",".join(["?"] * len(compound_names))

    # Trial context
    trial_ctx_rows = con.execute(f"""
        SELECT compound_name, trial_context
        FROM compounds
        WHERE compound_name IN ({placeholders})
          AND trial_context IS NOT NULL
    """, compound_names).fetchall()

    # PubMed abstract titles
    pubmed_rows = con.execute(f"""
        SELECT compound_name, pmid, title
        FROM compound_abstracts
        WHERE compound_name IN ({placeholders})
        ORDER BY compound_name, match_score DESC
    """, compound_names).fetchall()

    trial_ctx = defaultdict(str)
    for name, ctx in trial_ctx_rows:
        trial_ctx[name] = ctx or ""

    pubmed_ctx = defaultdict(list)
    for name, pmid, title in pubmed_rows:
        if len(pubmed_ctx[name]) < 2:
            pubmed_ctx[name].append(f"[PMID:{pmid}] {title}")

    lines = ["=== CONTEXT FOR THIS BATCH ==="]
    for name in compound_names:
        tc = trial_ctx.get(name, "")
        pm = " | ".join(pubmed_ctx.get(name, []))
        if tc or pm:
            lines.append(f"{name}:")
            if tc:
                lines.append(f"  TRIAL: {tc[:400]}")
            if pm:
                lines.append(f"  PUBMED: {pm}")

    return "\n".join(lines) if len(lines) > 1 else "No context available for this batch."


def clean_batch_txgemma(compound_names: list, con) -> list:
    """
    Send a batch of compound names to TxGemma 27B for cleaning.
    Uses LM Studio REST API (OpenAI-compatible /v1/chat/completions).
    Returns list of result dicts.
    """
    context_str = get_context_for_batch(compound_names, con)

    user_message = f"""{context_str}

Normalize these intervention strings. Return a JSON array only — no preamble, no markdown fences.
Each object must have exactly these keys:
  input, canonical_name (string or null), exclude (boolean),
  exclude_reason (string or null), confidence ("high"|"medium"|"low")

Compounds to clean:
{json.dumps(compound_names, indent=2)}"""

    payload = {
        "model"   : LLM_MODEL,
        "messages": [
            {"role": "system", "content": CLEANING_SYSTEM_PROMPT},
            {"role": "user",   "content": user_message},
        ],
        "temperature": 0.05,
        "max_tokens" : 4096,
        "stream"     : False,
    }

    response = requests.post(LLM_ENDPOINT, json=payload, timeout=300)
    response.raise_for_status()

    raw = response.json()["choices"][0]["message"]["content"].strip()

    # Strip markdown fences if model adds them despite instructions
    if raw.startswith("```"):
        raw = re.sub(r'^```(?:json)?\s*', '', raw)
        raw = re.sub(r'\s*```\s*$', '', raw)
    raw = raw.strip()

    results = json.loads(raw)

    # Normalize confidence — guard against model returning integers
    for r in results:
        c = r.get("confidence")
        if c in (1, "1", True):
            r["confidence"] = "high"
        elif c in (0, "0", False):
            r["confidence"] = "low"
        elif c not in ("high", "medium", "low"):
            r["confidence"] = "medium"

    return results


def run_txgemma_step(search_term: str, batch_size: int = LLM_BATCH_SIZE) -> dict:
    """
    Run Step 9: TxGemma 27B cleaning with per-batch timing + rolling ETA.
    Safe to re-run — skips compounds already cleaned.
    """
    term = search_term.lower().strip()
    con  = sqlite3.connect(SQLITE_DB)

    compounds = pd.read_sql("""
        SELECT compound_name FROM compounds
        WHERE search_term = ? AND canonical_name IS NULL
        ORDER BY compound_name
    """, con, params=(term,))

    queue = compounds["compound_name"].tolist()

    if not queue:
        print("✅ No compounds remaining for TxGemma — all handled by earlier steps")
        con.close()
        return {"txgemma_cleaned": 0, "txgemma_excluded": 0}

    # Auto-reduce batch for very long strings
    avg_len = sum(len(c) for c in queue) / max(len(queue), 1)
    if avg_len > 60:
        batch_size = max(5, batch_size // 2)
        print(f"⚠️  Long strings (avg {avg_len:.0f} chars) — reducing batch to {batch_size}")

    total_batches   = (len(queue) + batch_size - 1) // batch_size
    cleaned_count   = 0
    excluded_count  = 0
    error_count     = 0
    batch_times     = []

    print(f"🤖 TxGemma: {len(queue)} compounds in {total_batches} batches (size={batch_size})")

    for batch_num in range(total_batches):
        batch = queue[batch_num * batch_size : (batch_num + 1) * batch_size]

        # Rolling ETA
        if batch_times:
            avg_batch_time = sum(batch_times) / len(batch_times)
            remaining      = total_batches - batch_num
            eta_sec        = avg_batch_time * remaining
            eta_str        = f"ETA {eta_sec:.0f}s"
        else:
            eta_str = "ETA unknown"

        print(f"\n📦 Batch {batch_num+1}/{total_batches} ({len(batch)} compounds) | {eta_str}")

        batch_t0 = time.time()
        try:
            results = clean_batch_txgemma(batch, con)
        except json.JSONDecodeError as e:
            print(f"   ⚠️  JSON parse error: {e} — logging and skipping batch")
            error_count += 1
            _log_batch_error(con, batch_num, str(e), time.time() - batch_t0)
            continue
        except Exception as e:
            print(f"   ⚠️  Batch error: {e}")
            error_count += 1
            _log_batch_error(con, batch_num, str(e), time.time() - batch_t0)
            continue

        batch_elapsed = time.time() - batch_t0
        batch_times.append(batch_elapsed)

        now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        for result in results:
            try:
                input_name     = result.get("input", "")
                canonical_name = result.get("canonical_name") or None
                exclude        = 1 if result.get("exclude") else 0
                exclude_reason = result.get("exclude_reason") or None
                confidence     = result.get("confidence", "low")

                if isinstance(canonical_name, list):
                    canonical_name = canonical_name[0] if canonical_name else None

                # --- Sponsor code override ---
                # If TxGemma wants to exclude something that looks like a
                # sponsor code, override and leave it NULL for human review
                SPONSOR_CODE_RE = re.compile(
                    r'^[A-Z]{2,6}[-_]\d{3,}|'   # CLN-619, JNJ-12345
                    r'^[A-Z]{2,6}\d{3,}|'        # AMG701, BMS123456
                    r'^[A-Z]{1,4}-[A-Z]{2,4}-\d{3,}|'  # ABBV-CLS-484
                    r'^[A-Z]{2,6}-[A-Z0-9]{2,8}-\d{2,}',  # compound codes
                    re.IGNORECASE
                )

                if exclude and SPONSOR_CODE_RE.match(input_name.strip()):
                    # Override — pass to human review instead of excluding
                    canonical_name = None
                    exclude        = 0
                    exclude_reason = None
                    confidence     = None
                    print(f"   🔄 Sponsor code override: '{input_name}' → human review")
                elif exclude:
                    canonical_name = "EXCLUDED"
                    excluded_count += 1
                elif canonical_name:
                    cleaned_count += 1

                con.execute("""
                    UPDATE compounds
                    SET canonical_name = ?,
                        exclude        = ?,
                        exclude_reason = ?,
                        confidence     = ?,
                        cleaned_by     = 'txgemma'
                    WHERE compound_name = ? AND search_term = ?
                """, (canonical_name, exclude, exclude_reason,
                      confidence, input_name, term))

                # Log to cleaning_log
                con.execute("""
                    INSERT INTO cleaning_log (
                        run_id, prompt_version, batch_num,
                        compound_name, raw_response,
                        success, elapsed_sec, created_at
                    ) VALUES (?, ?, ?, ?, ?, 1, ?, ?)
                """, (
                    RUN_ID, PROMPT_VERSION, batch_num + 1,
                    input_name, json.dumps(result),
                    batch_elapsed, now
                ))

            except Exception as e:
                print(f"   ⚠️  Result write error for '{result.get('input', '?')}': {e}")

        con.commit()
        rate = len(batch) / max(batch_elapsed, 0.01)
        print(f"   ✅ Done in {batch_elapsed:.1f}s ({rate:.1f} compounds/s)")

    con.close()

    print(f"\n🤖 TxGemma summary:")
    print(f"   Cleaned (kept)   : {cleaned_count}")
    print(f"   Excluded         : {excluded_count}")
    print(f"   Batch errors     : {error_count}")
    return {"txgemma_cleaned": cleaned_count, "txgemma_excluded": excluded_count}


def _log_batch_error(con, batch_num, error_msg, elapsed):
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    con.execute("""
        INSERT INTO cleaning_log (
            run_id, prompt_version, batch_num, success, error_msg, elapsed_sec, created_at
        ) VALUES (?, ?, ?, 0, ?, ?, ?)
    """, (RUN_ID, PROMPT_VERSION, batch_num + 1, error_msg, elapsed, now))
    con.commit()


with step_timer("Step 9: TxGemma 27B cleaning"):
    txgemma_stats = run_txgemma_step(disease_input.value)
efficacy_snapshot("after TxGemma")

⏳ Starting: Step 9: TxGemma 27B cleaning
🤖 TxGemma: 34 compounds in 3 batches (size=15)

📦 Batch 1/3 (15 compounds) | ETA unknown
   🔄 Sponsor code override: 'CH505 TF mRNA-gp160' → human review
   🔄 Sponsor code override: 'CH505M5 N197D mRNA-gp160' → human review
   🔄 Sponsor code override: 'DV700P-RNA' → human review
   🔄 Sponsor code override: 'DV701B1.1-RNA' → human review
   🔄 Sponsor code override: 'GS-1720' → human review
   🔄 Sponsor code override: 'GS-4182' → human review
   🔄 Sponsor code override: 'GS-4182 FDC' → human review
   ✅ Done in 139.0s (0.1 compounds/s)

📦 Batch 2/3 (15 compounds) | ETA 278s
   ✅ Done in 125.1s (0.1 compounds/s)

📦 Batch 3/3 (4 compounds) | ETA 132s
   ✅ Done in 37.4s (0.1 compounds/s)

🤖 TxGemma summary:
   Cleaned (kept)   : 18
   Excluded         : 7
   Batch errors     : 0
✅ Step 9: TxGemma 27B cleaning complete — 301.7s

📊 Efficacy snapshot [after TxGemma]
   Total compounds  : 69
   Resolved (kept)  : 41
   Excluded         : 19
   Unresolved

In [28]:
# =============================================================================
# Cell 20 — Step 10: Post-cleaning Canonical Deduplication
# =============================================================================
# NEW in v3:
#   After LLM cleaning, multiple input strings may map to the same
#   canonical_name (e.g. 'bortezomib' and 'bortezomib (VELCADE)' both → 'bortezomib').
#   This step identifies canonical duplicates and merges them:
#     - One row is kept as the canonical representative
#     - Duplicates are marked exclude=2 (dedup) with a pointer to canonical
#     - trial_compounds junction is updated to point to canonical
# =============================================================================

def run_canonical_dedup(search_term: str) -> dict:
    """
    Merge canonical name duplicates after cleaning.
    Keeps one row per canonical_name (preferring shortest compound_name).
    Updates trial_compounds to use canonical representative.
    Returns dedup statistics.
    """
    term = search_term.lower().strip()
    con  = sqlite3.connect(SQLITE_DB)

    # Find all canonical names that have > 1 compound_name mapped to them
    dupes = pd.read_sql("""
        SELECT canonical_name, COUNT(*) AS cnt,
               MIN(LENGTH(compound_name)) AS min_len
        FROM compounds
        WHERE search_term    = ?
          AND canonical_name IS NOT NULL
          AND canonical_name != 'EXCLUDED'
          AND exclude        = 0
        GROUP BY canonical_name
        HAVING COUNT(*) > 1
    """, con, params=(term,))

    merged = 0
    dedup_map = {}  # old_compound_name → canonical representative compound_name

    for _, row in dupes.iterrows():
        canon = row["canonical_name"]

        # Get all compound names mapped to this canonical
        variants = pd.read_sql("""
            SELECT compound_name FROM compounds
            WHERE search_term    = ?
              AND canonical_name = ?
              AND exclude        = 0
            ORDER BY LENGTH(compound_name) ASC
        """, con, params=(term, canon))

        names = variants["compound_name"].tolist()
        # Keep the shortest name as canonical representative
        representative = names[0]
        duplicates     = names[1:]

        for dup in duplicates:
            # Update trial_compounds junction table
            # If the canonical already exists for that nct_id+source, delete
            # the duplicate row rather than updating (avoids UNIQUE constraint)
            existing = con.execute("""
                SELECT 1 FROM trial_compounds
                WHERE compound_name = ?
            """, (representative,)).fetchone()

            if existing:
                # Canonical already represented — just delete the duplicate row
                con.execute("""
                    DELETE FROM trial_compounds
                    WHERE compound_name = ?
                """, (dup,))
            else:
                # Safe to rename
                con.execute("""
                    UPDATE trial_compounds
                    SET compound_name = ?
                    WHERE compound_name = ?
                """, (representative, dup))

            # Mark the duplicate as deduped
            con.execute("""
                UPDATE compounds
                SET exclude        = 2,
                    exclude_reason = ?,
                    cleaned_by     = 'dedup'
                WHERE compound_name = ? AND search_term = ?
            """, (f"canonical duplicate of '{representative}'", dup, term))

            dedup_map[dup] = representative
            merged += 1

    con.commit()

    total_unique = con.execute("""
        SELECT COUNT(DISTINCT canonical_name) FROM compounds
        WHERE search_term = ? AND exclude = 0
    """, (term,)).fetchone()[0]

    con.close()

    print(f"   Canonical duplicates merged : {merged}")
    print(f"   Unique canonical names      : {total_unique}")
    if dedup_map:
        print("   Sample merges:")
        for old, new in list(dedup_map.items())[:5]:
            print(f"     '{old}' → '{new}'")

    return {"dedup_merged": merged, "unique_canonical": total_unique}


with step_timer("Step 10: Canonical deduplication"):
    dedup_stats = run_canonical_dedup(disease_input.value)
efficacy_snapshot("after dedup")

⏳ Starting: Step 10: Canonical deduplication
   Canonical duplicates merged : 18
   Unique canonical names      : 23
   Sample merges:
     'Bariticinib' → 'Barticinib'
     'Baricitinib' → 'Barticinib'
     'Bictegravir' → 'Biktarvy'
     'CAB ULA' → 'CAB LA'
     'Rilpivirine' → 'CAB LA'
✅ Step 10: Canonical deduplication complete — 0.0s

📊 Efficacy snapshot [after dedup]
   Total compounds  : 69
   Resolved (kept)  : 23
   Excluded         : 19
   Unresolved       : 9
   Low confidence   : 2


In [29]:
# =============================================================================
# Cell 21 — Step 11: Human Review GUI (or fallback DataFrame)
# =============================================================================
# Presents NULL (unresolved) and low-confidence keeps for human review.
# Falls back to a plain DataFrame display if ipywidgets is unavailable.
# Target: < 30 compounds in this queue.
# =============================================================================

def load_review_queue(search_term: str) -> pd.DataFrame:
    """Load compounds requiring human review: NULL or low-confidence."""
    term = search_term.lower().strip()
    con  = sqlite3.connect(SQLITE_DB)
    df   = pd.read_sql("""
        SELECT compound_name, canonical_name, confidence,
               ner_class, cleaned_by, trial_context,
               exclude, exclude_reason
        FROM compounds
        WHERE search_term = ?
          AND (
                (canonical_name IS NULL)
                OR
                (confidence = 'low' AND exclude = 0)
              )
        ORDER BY compound_name
    """, con, params=(term,))
    con.close()
    return df


def apply_human_decision(compound_name: str, search_term: str,
                         canonical: str, action: str):
    """
    Write a human review decision back to the database.
    action: 'keep' | 'exclude'
    """
    term = search_term.lower().strip()
    con  = sqlite3.connect(SQLITE_DB)
    if action == "exclude":
        con.execute("""
            UPDATE compounds
            SET canonical_name = 'EXCLUDED',
                exclude        = 1,
                exclude_reason = 'human review — excluded',
                confidence     = 'high',
                cleaned_by     = 'human'
            WHERE compound_name = ? AND search_term = ?
        """, (compound_name, term))
    else:  # keep
        final_canonical = canonical.strip().lower() if canonical.strip() else compound_name.lower()
        con.execute("""
            UPDATE compounds
            SET canonical_name = ?,
                confidence     = 'high',
                cleaned_by     = 'human',
                exclude        = 0
            WHERE compound_name = ? AND search_term = ?
        """, (final_canonical, compound_name, term))
    con.commit()
    con.close()


# ---------------------------------------------------------------------------
# Widget UI
# ---------------------------------------------------------------------------
review_df = load_review_queue(disease_input.value)
print(f"\n🔍 Human review queue: {len(review_df)} compounds")
if len(review_df) > 30:
    print(f"⚠️  Queue > 30 — review earlier cleaning steps for missed exclusions")

try:
    if review_df.empty:
        display(HTML("<b>✅ No compounds requiring human review.</b>"))
    else:
        # Build widget per compound
        review_widgets = []
        review_state   = {}  # compound_name → (canonical_text_widget, action_toggle)

        for _, row in review_df.iterrows():
            cname  = row["compound_name"]
            ccanon = row["canonical_name"] or ""
            cconf  = row["confidence"]     or "NULL"
            ctx    = (row["trial_context"] or "")[:200]

            label = widgets.HTML(
                f"<b>{cname}</b> "
                f"<span style='color:grey'>confidence={cconf}</span><br>"
                f"<small style='color:#555'>{ctx}</small>"
            )
            canon_input = widgets.Text(
                value=ccanon,
                placeholder="Canonical name (or leave blank to exclude)",
                layout=widgets.Layout(width="350px")
            )
            action_toggle = widgets.ToggleButtons(
                options=["keep", "exclude"],
                value="keep" if ccanon else "exclude",
                button_style="",
                layout=widgets.Layout(width="220px")
            )
            review_state[cname] = (canon_input, action_toggle)
            review_widgets.append(
                widgets.HBox([label, canon_input, action_toggle],
                             layout=widgets.Layout(
                                 border="1px solid #ddd",
                                 padding="6px", margin="2px 0"
                             ))
            )

        save_btn    = widgets.Button(description="Save All Decisions",
                                     button_style="success",
                                     layout=widgets.Layout(width="200px"))
        review_out  = widgets.Output()

        def on_save(b):
            with review_out:
                review_out.clear_output()
                term = disease_input.value
                saved = 0
                for cname, (canon_w, action_w) in review_state.items():
                    apply_human_decision(
                        cname, term,
                        canon_w.value, action_w.value
                    )
                    saved += 1
                print(f"✅ {saved} decisions saved")
                efficacy_snapshot("after human review")

        save_btn.on_click(on_save)

        display(widgets.VBox([
            widgets.HTML(f"<h3>👤 Human Review — {len(review_df)} compounds</h3>"),
            *review_widgets,
            save_btn,
            review_out
        ]))

except Exception as e:
    print(f"⚠️  Widget error ({e}) — falling back to DataFrame display")
    display(review_df)


🔍 Human review queue: 9 compounds


In [30]:
# =============================================================================
# Cell 22 — Cleaning Efficacy Summary with Timing Breakdown
# =============================================================================

pipeline_elapsed = time.time() - PIPELINE_START

con = sqlite3.connect(SQLITE_DB)

# Resolution counts by cleaned_by step
by_step = pd.read_sql("""
    SELECT
        COALESCE(cleaned_by, 'unresolved') AS step,
        COUNT(*)                           AS compounds,
        SUM(CASE WHEN exclude = 0 THEN 1 ELSE 0 END) AS kept,
        SUM(CASE WHEN exclude = 1 THEN 1 ELSE 0 END) AS excluded
    FROM compounds
    GROUP BY step
    ORDER BY compounds DESC
""", con)

# Confidence breakdown (kept compounds only)
by_conf = pd.read_sql("""
    SELECT confidence, COUNT(*) AS count
    FROM compounds
    WHERE exclude = 0
    GROUP BY confidence
""", con)

# NER class breakdown
by_ner = pd.read_sql("""
    SELECT COALESCE(ner_class, 'unclassified') AS ner_class,
           COUNT(*) AS count
    FROM compounds
    GROUP BY ner_class
    ORDER BY count DESC
""", con)

con.close()

print("=" * 60)
print(f"  CTIP Pipeline Efficacy Summary — {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"  Run ID     : {RUN_ID}")
print(f"  Search term: {disease_input.value}")
print("=" * 60)

print("\n📊 Resolution by Step:")
display(by_step)

print("\n📊 Confidence Breakdown (kept compounds):")
display(by_conf)

print("\n📊 NER Class Breakdown:")
display(by_ner)

print("\n⏱  Timing Breakdown:")
for step, elapsed in step_timings.items():
    print(f"   {step:<45} {elapsed:>7.1f}s")
print(f"   {'Total pipeline wall time':<45} {pipeline_elapsed:>7.1f}s")

# Final counts
con = sqlite3.connect(SQLITE_DB)
final = con.execute("""
    SELECT
        COUNT(*)                                              AS total,
        SUM(CASE WHEN exclude = 0 AND canonical_name IS NOT NULL
                      AND canonical_name != 'EXCLUDED' THEN 1 ELSE 0 END) AS final_kept,
        SUM(CASE WHEN exclude IN (1, 2) THEN 1 ELSE 0 END)  AS total_excluded,
        SUM(CASE WHEN canonical_name IS NULL THEN 1 ELSE 0 END) AS still_unresolved
    FROM compounds
""").fetchone()
con.close()

total, kept, excluded, unresolved = final
print(f"\n{'─' * 60}")
print(f"  Input compounds    : {total}")
print(f"  Final kept         : {kept}")
print(f"  Total excluded     : {excluded}")
print(f"  Still unresolved   : {unresolved}")
if total > 0:
    print(f"  Resolution rate    : {(kept + excluded) / total * 100:.1f}%")
print("=" * 60)

  CTIP Pipeline Efficacy Summary — 2026-03-14 16:38
  Run ID     : run_20260314_152408_f93208
  Search term: hiv

📊 Resolution by Step:


,step,compounds,kept,excluded
0,dedup,18,0,0
1,txgemma,15,8,7
2,rxnorm,14,14,0
3,pre-filter,12,0,12
4,human,9,9,0
5,chembl,1,1,0



📊 Confidence Breakdown (kept compounds):


,confidence,count
0,high,29
1,medium,3



📊 NER Class Breakdown:


,ner_class,count
0,unclassified,35
1,UNKNOWN,34



⏱  Timing Breakdown:
   Step 1: Deterministic pre-filter                  0.0s
   Step 2: RxNorm lookup                            26.4s
   Step 3: ChEMBL lookup                           464.4s
   Step 4: Fuzzy INN matching                        0.2s
   Step 5: CAR-T classification                      0.0s
   Step 6: NER classification                        0.0s
   Step 7: Trial context enrichment                  0.0s
   Step 7b: PubMed corpus collection               567.4s
   Step 8: PubMed context                            2.0s
   Step 9: TxGemma 27B cleaning                    301.7s
   Step 10: Canonical deduplication                  0.0s
   Total pipeline wall time                       4466.2s

────────────────────────────────────────────────────────────
  Input compounds    : 69
  Final kept         : 32
  Total excluded     : 37
  Still unresolved   : 0
  Resolution rate    : 100.0%


In [31]:
# =============================================================================
# Cell 23 — Final Compounds Inspection + Junction Table Sample
# =============================================================================

con = sqlite3.connect(SQLITE_DB)
term = disease_input.value.lower().strip()

print("--- Final Kept Compounds (first 30) ---")
kept = pd.read_sql("""
    SELECT compound_name, canonical_name, source,
           intervention_type, ner_class, confidence, cleaned_by
    FROM compounds
    WHERE search_term = ?
      AND exclude     = 0
      AND canonical_name IS NOT NULL
      AND canonical_name != 'EXCLUDED'
    ORDER BY canonical_name
    LIMIT 30
""", con, params=(term,))
display(kept)

print("\n--- Unresolved / Review Queue ---")
unresolved = pd.read_sql("""
    SELECT compound_name, ner_class, confidence, trial_context
    FROM compounds
    WHERE search_term = ? AND canonical_name IS NULL
    ORDER BY compound_name
""", con, params=(term,))
print(f"Count: {len(unresolved)}")
display(unresolved)

print("\n--- Junction Table Sample (trial → canonical compound) ---")
junction = pd.read_sql("""
    SELECT tc.nct_id, t.brief_title, tc.compound_name, c.canonical_name,
           c.confidence, tc.source
    FROM trial_compounds tc
    JOIN trials   t ON tc.nct_id       = t.nct_id
    JOIN compounds c ON tc.compound_name = c.compound_name
    WHERE c.exclude = 0
      AND c.canonical_name IS NOT NULL
    LIMIT 15
""", con)
display(junction)

con.close()

--- Final Kept Compounds (first 30) ---


,compound_name,canonical_name,source,intervention_type,ner_class,confidence,cleaned_by
0,Pyridoxine Hydrochloride,ascorbic acid,intervention,DRUG,NaN,high,rxnorm
1,Azithromycin,azithromycin,intervention,DRUG,NaN,high,rxnorm
2,Barticinib,baricitinib,summary,,UNKNOWN,high,txgemma
3,Biktarvy,bictegravir,summary,,NaN,high,rxnorm
4,CAB LA,cabotegravir,intervention,DRUG,UNKNOWN,medium,txgemma
5,CH505 TF mRNA-gp160,ch505 tf mrna-gp160,intervention,BIOLOGICAL,UNKNOWN,high,human
6,CH505M5 N197D mRNA-gp160,ch505m5 n197d mrna-gp160,intervention,BIOLOGICAL,UNKNOWN,high,human
7,tenofovir alafenamide,cobicistat,intervention,DRUG,NaN,high,rxnorm
8,Dapagliflozin,dapagliflozin,intervention,DRUG,NaN,high,rxnorm
9,DV700P-RNA,dv700p-rna,intervention,BIOLOGICAL,UNKNOWN,high,human



--- Unresolved / Review Queue ---
Count: 0


,compound_name,ner_class,confidence,trial_context



--- Junction Table Sample (trial → canonical compound) ---


,nct_id,brief_title,compound_name,canonical_name,confidence,source
0,NCT07028385,Safety and Impact of Baricitinib on Cell Suriv...,Barticinib,baricitinib,high,summary
1,NCT06919016,A Study to Evaluate the Safety and Immunogenic...,DV700P-RNA,dv700p-rna,high,intervention
2,NCT06919016,A Study to Evaluate the Safety and Immunogenic...,DV701B1.1-RNA,dv701b1.1-rna,high,intervention
3,NCT07055828,Nebulized vs. IV Ketamine in Preventing Shiver...,Ketamine,ketamine,high,intervention
4,NCT06970223,A Study to Investigate if Long Acting Cabotegr...,Lenacapavir,lenacapavir,high,title
5,NCT06580587,A Study of MK-8527 in Healthy Lactating Female...,MK-8527,mk-8527,medium,intervention
6,NCT06580587,A Study of MK-8527 in Healthy Lactating Female...,MK-8527,mk-8527,medium,title
7,NCT06580587,A Study of MK-8527 in Healthy Lactating Female...,MK-8527,mk-8527,medium,summary
8,NCT06891066,A Study of Islatravir (ISL) and Ulonivirine (U...,MK-8591B,islatravir + ulonivirine,high,summary
9,NCT06786520,"A Study to Evaluate the Pharmacokinetics, Safe...",CAB LA,cabotegravir,medium,intervention


In [ ]:
# =============================================================================
# Cell 24 — Reset Utility (commented out — run manually only)
# =============================================================================
# Uncomment and run to fully reset the database and restart the pipeline.
# DESTRUCTIVE — all cleaning progress will be lost.

# def full_reset():
#     """Drop all tables and reinitialise schema. DESTRUCTIVE."""
#     import os
#     if os.path.exists(SQLITE_DB):
#         os.remove(SQLITE_DB)
#         print(f"🗑  Removed {SQLITE_DB}")
#     init_sqlite()
#     print("✅ Database reset complete — re-run from Cell 5")
#
# full_reset()


# ---------------------------------------------------------------------------
# Partial reset — clear only cleaning columns (keep registry)
# Useful when re-running cleaning steps with updated prompts or rules
# ---------------------------------------------------------------------------
# def reset_cleaning_only(search_term: str):
#     """Reset canonical_name and all cleaning columns for a search term."""
#     term = search_term.lower().strip()
#     con  = sqlite3.connect(SQLITE_DB)
#     con.execute("""
#         UPDATE compounds
#         SET canonical_name = NULL,
#             exclude        = 0,
#             exclude_reason = NULL,
#             confidence     = NULL,
#             cleaned_by     = NULL,
#             ner_class      = NULL,
#             ner_confidence = NULL,
#             trial_context  = NULL,
#             pubmed_context = NULL
#         WHERE search_term = ?
#     """, (term,))
#     con.execute("DELETE FROM cleaning_log WHERE run_id LIKE ?" , (f"%{term}%",))
#     con.execute("DELETE FROM compound_abstracts")
#     con.commit()
#     con.close()
#     print(f"✅ Cleaning reset for '{term}' — registry preserved")
#
# reset_cleaning_only(disease_input.value)

# print("✅ Reset utilities defined (commented out)")
# print("   Uncomment full_reset() or reset_cleaning_only() to use")

In [ ]:
# # Quick LM Studio connectivity check
# import requests, json

# try:
#     resp = requests.post(
#         LLM_ENDPOINT,
#         json={
#             "model"   : LLM_MODEL,
#             "messages": [{"role": "user", "content": "Reply with the single word: ready"}],
#             "max_tokens": 10,
#         },
#         timeout=30
#     )
#     resp.raise_for_status()
#     print("✅ LM Studio connected")
#     print(f"   Response: {resp.json()['choices'][0]['message']['content'].strip()}")
# except Exception as e:
#     print(f"❌ Connection failed: {e}")

In [ ]:
# import sqlite3, gc

# # Force close any open connections
# gc.collect()

# # Open and immediately close to verify DB is accessible
# try:
#     con = sqlite3.connect(SQLITE_DB, timeout=5)
#     con.execute("SELECT 1")
#     con.close()
#     print("✅ Database is accessible")
# except Exception as e:
#     print(f"❌ Still locked: {e}")

In [ ]:
# import sqlite3
# con = sqlite3.connect(SQLITE_DB)
# con.execute("""
#     UPDATE compounds
#     SET canonical_name = NULL,
#         confidence     = NULL,
#         cleaned_by     = NULL,
#         exclude        = NULL
#     WHERE search_term = ?
# """, (disease_input.value.lower().strip(),))
# con.commit()
# con.close()
# print("✅ All cleaning results cleared — ready to re-run from Cell 12")

In [ ]:
# # Check scispaCy and bc5cdr model
# try:
#     import spacy
#     print(f"✅ spacy version: {spacy.__version__}")
# except ImportError:
#     print("❌ spacy not installed")

# try:
#     import scispacy
#     print(f"✅ scispacy version: {scispacy.__version__}")
# except ImportError:
#     print("❌ scispacy not installed")

# try:
#     import spacy
#     nlp = spacy.load("en_ner_bc5cdr_md")
#     print("✅ en_ner_bc5cdr_md model loaded")
# except OSError:
#     print("❌ en_ner_bc5cdr_md model not found — needs installing")
# except Exception as e:
#     print(f"❌ Unexpected error: {e}")

In [ ]:
# import subprocess, sys

# print("Installing spacy 3.7.5 (Python 3.7 compatible)...")
# result = subprocess.run(
#     [sys.executable, "-m", "pip", "install",
#      "spacy==3.7.5",
#      "--only-binary=:all:"],
#     capture_output=True, text=True
# )
# print(result.stdout[-500:] if result.stdout else "")
# print(result.stderr[-300:] if result.stderr else "")

# print("\nInstalling scispacy 0.5.4...")
# result = subprocess.run(
#     [sys.executable, "-m", "pip", "install",
#      "scispacy==0.5.4",
#      "--only-binary=:all:"],
#     capture_output=True, text=True
# )
# print(result.stdout[-500:] if result.stdout else "")
# print(result.stderr[-300:] if result.stderr else "")

# print("\nInstalling en_ner_bc5cdr_md model...")
# result = subprocess.run(
#     [sys.executable, "-m", "pip", "install",
#      "https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz",
#      "--no-deps"],
#     capture_output=True, text=True
# )
# print(result.stdout[-500:] if result.stdout else "")
# print(result.stderr[-300:] if result.stderr else "")

# print("\nDone. Re-run diagnostic to confirm.")

In [ ]:
# try:
#     import spacy
#     print(f"✅ spacy version: {spacy.__version__}")
# except ImportError:
#     print("❌ spacy not installed")

# try:
#     import scispacy
#     print(f"✅ scispacy version: {scispacy.__version__}")
# except ImportError:
#     print("❌ scispacy not installed")

# try:
#     import spacy
#     nlp = spacy.load("en_ner_bc5cdr_md")
#     print("✅ en_ner_bc5cdr_md model loaded")
# except OSError:
#     print("❌ en_ner_bc5cdr_md model not found")
# except Exception as e:
#     print(f"❌ Unexpected error: {e}")

In [ ]:
import sqlite3
con = sqlite3.connect("trials_v3.db")

rows = con.execute("""
    SELECT compound_name, canonical_name, exclude_reason, cleaned_by
    FROM compounds
    WHERE exclude = 1
    AND search_term = 'parkinsons'
    ORDER BY cleaned_by, compound_name
""").fetchall()

print(f"{'Compound':<40} {'Canonical':<30} {'Excluded By':<15} {'Reason'}")
print("-" * 110)
for row in rows:
    print(f"  {str(row[0]):<38} {str(row[1] or ''):<30} {str(row[3] or ''):<15} {str(row[2] or '')}")

con.close()

In [ ]:
import sqlite3
con = sqlite3.connect(SQLITE_DB)

rows = con.execute("""
    SELECT compound_name, exclude_reason
    FROM compounds
    WHERE search_term = ?
    AND cleaned_by = 'pre-filter'
    AND exclude = 1
    ORDER BY compound_name
""", (disease_input.value.lower().strip(),)).fetchall()

for row in rows:
    print(f"  {row[0]:<40} {row[1]}")

con.close()

In [ ]:
import sqlite3
con = sqlite3.connect(SQLITE_DB)

rows = con.execute("""
    SELECT compound_name, canonical_name, exclude_reason, cleaned_by
    FROM compounds
    WHERE search_term = ?
    AND exclude = 1
    ORDER BY cleaned_by, compound_name
""", (disease_input.value.lower().strip(),)).fetchall()

print(f"{'Compound':<40} {'Canonical':<15} {'Cleaned By':<15} {'Reason'}")
print("-" * 100)
for row in rows:
    print(f"  {str(row[0]):<38} {str(row[1] or ''):<15} {str(row[3] or ''):<15} {str(row[2] or '')}")

con.close()

In [ ]:
import sqlite3
con = sqlite3.connect(SQLITE_DB)

rows = con.execute("""
    SELECT compound_name, ner_class, 
           CASE WHEN pubmed_context IS NOT NULL THEN 'yes' ELSE 'no' END as has_context
    FROM compounds
    WHERE search_term = ?
    AND exclude = 0
    AND canonical_name IS NULL
    ORDER BY ner_class, compound_name
""", (disease_input.value.lower().strip(),)).fetchall()

for row in rows:
    print(f"  {row[0]:<40} {row[1]:<15} {row[2]}")

con.close()

In [19]:
# Quick LM Studio connectivity check
import requests, json

try:
    resp = requests.post(
        LLM_ENDPOINT,
        json={
            "model"   : LLM_MODEL,
            "messages": [{"role": "user", "content": "Reply with the single word: ready"}],
            "max_tokens": 10,
        },
        timeout=30
    )
    resp.raise_for_status()
    print("✅ LM Studio connected")
    print(f"   Response: {resp.json()['choices'][0]['message']['content'].strip()}")
except Exception as e:
    print(f"❌ Connection failed: {e}")

✅ LM Studio connected
   Response: ready
